# Quantum-Dominant Parameter-Matched U-Net for Speech Enhancement

This notebook is a research-ready, resume-safe investigation of **multi-location exact variational quantum feature mixing** in a lightweight complex-spectral U-Net. It compares three arms:

1. `baseline`: the shared U-Net without conditioning mixers;
2. `classical`: the same U-Net with parameter-matched bias-free MLP mixers;
3. `qnn`: the same U-Net with exact differentiable six-qubit state-vector VQCs.

The classical and quantum arms use identical statistics, projections, connections, modulation equations, data, initialization, losses, schedules, and evaluation files. Only their 36- or 72-parameter mixer cores differ.

> **Scientific constraint.** This notebook gives the QNN a strong but fair opportunity; it cannot guarantee a quantum win. Screening uses training and validation data only. The locked test manifest is evaluated only after a configuration is frozen. Failed, tied, partial, and successful outcomes are all retained.

> **Known split limitation.** The repository's fixed 15,000/2,000/2,000 splits contain all 23 speakers and all 11 noise labels in every split. They are preserved for comparability, but the main results are **not speaker-independent** and do not constitute an unseen-noise evaluation.


## 0 — Environment and execution policy

Run the cells in order. Installation, screening, confirmation, locked-test evaluation, and secondary studies are opt-in. No full training starts merely by opening or running the notebook top-to-bottom with the default switches.


In [ ]:
# Uncomment in a fresh environment if required.
# %pip install -q "numpy<2" librosa==0.11.0 soundfile pandas tqdm scipy pesq pystoi matplotlib
# Optional independent circuit verification only:
%pip install -q pennylane==0.45.0


## 1 — Imports, configuration, paths, and reproducibility


In [ ]:
import os
import gc
import io
import json
import math
import time
import copy
import random
import hashlib
import warnings
import tempfile
from dataclasses import dataclass, asdict, replace
from pathlib import Path
from datetime import datetime, timezone
from contextlib import nullcontext
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
try:
    import librosa
except ImportError:
    librosa = None
import soundfile as sf
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler

warnings.filterwarnings("ignore", category=UserWarning)


def locate_iteration_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "misc" / "iteration",
        Path("."),
    ]
    for candidate in candidates:
        if (candidate / "my_speech_enhancement_dataset_v3").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate misc/iteration/my_speech_enhancement_dataset_v3")


ITERATION_DIR = locate_iteration_dir()


@dataclass(frozen=True)
class Config:
    # Audio and data
    sample_rate: int = 16_000
    crop_seconds: int = 8
    n_fft: int = 320
    win_length: int = 320
    hop_length: int = 160
    dataset_root: str = str(ITERATION_DIR / "my_speech_enhancement_dataset_v3")
    run_root: str = str(ITERATION_DIR / "runs" / "qunet_multivqc")
    batch_size: int = 8
    num_workers: int = 2
    pin_memory: bool = True
    audio_audit_workers: int = 8
    full_audio_header_audit: bool = True

    # U-Net
    encoder_channels: tuple = (32, 64, 96, 128)
    blocks_per_scale: int = 2
    bottleneck_blocks: int = 3
    dropout: float = 0.08
    phase_residual_scale: float = 0.75
    complex_residual_scale: float = 0.10
    min_parameters: int = 500_000
    max_parameters: int = 800_000

    # Mixer candidate
    variant: str = "qnn"
    n_qubits: int = 6
    q_depth: int = 2
    entanglement: str = "ring"
    conditioning_layout: str = "bottleneck_and_two_skip_gates"
    data_reuploading: bool = True

    # Loss
    loss_complex_ri: float = 0.30
    loss_magnitude: float = 0.20
    loss_phase: float = 0.05
    loss_mrstft: float = 0.20
    loss_sisdr: float = 0.25
    low_snr_weighting: bool = True

    # Training
    shared_pretrain_epochs: int = 20
    shared_pretrain_patience: int = 6
    mixer_warmup_epochs: int = 8
    mixer_warmup_patience: int = 4
    joint_finetune_epochs: int = 35
    joint_finetune_patience: int = 7
    min_delta: float = 1e-4
    backbone_lr: float = 1e-4
    projection_lr: float = 2e-4
    core_lr: float = 5e-4
    weight_decay: float = 1e-4
    grad_clip_norm: float = 5.0
    use_amp: bool = True
    checkpoint_every_steps: int = 200
    evaluation_save_interval: int = 25

    # Protocol
    seeds: tuple = (17, 42, 123)
    screen_seed: int = 17
    screen_confirmation_seeds: tuple = (17, 42)
    screen_warmup_epochs: int = 4
    screen_joint_epochs: int = 10
    screening_validation_per_snr: int = 40
    snr_levels: tuple = (-5, 0, 5, 10, 15, 20)
    latency_rtf_limit: float = 1.0
    bootstrap_repeats: int = 2_000
    strict_determinism: bool = True

    # Secondary studies (off by default)
    reduced_data_fractions: tuple = (0.10, 0.25, 0.50, 1.00)
    held_out_noises: tuple = ("siren", "helicopter")
    external_manifest: str | None = None

    @property
    def crop_length(self):
        return self.sample_rate * self.crop_seconds

    @property
    def freq_bins(self):
        return self.n_fft // 2 + 1


CFG = Config()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = bool(CFG.use_amp and DEVICE.type == "cuda")
RUN_ROOT = Path(CFG.run_root)

# ============================================================================
# RUNTIME-ONLY PERFORMANCE SETTINGS
# ----------------------------------------------------------------------------
# IMPORTANT:
# These values deliberately live OUTSIDE Config so they do NOT change the
# configuration hash used by the auto-resume/checkpoint system.
#
# Keep CFG.batch_size=8 and CFG.num_workers=2 unchanged for the existing run.
# The DataLoader functions below use these runtime settings instead.
#
# Override from the shell/notebook only if needed, for example:
#   os.environ["QUNET_RUNTIME_NUM_WORKERS"] = "6"
# ============================================================================
_CPU_COUNT = os.cpu_count() or 4
RUNTIME_NUM_WORKERS = int(os.environ.get(
    "QUNET_RUNTIME_NUM_WORKERS",
    str(min(8, max(2, _CPU_COUNT // 2))),
))
RUNTIME_PREFETCH_FACTOR = int(os.environ.get("QUNET_RUNTIME_PREFETCH_FACTOR", "4"))
RUNTIME_PERSISTENT_WORKERS = bool(
    RUNTIME_NUM_WORKERS > 0
    and os.environ.get("QUNET_RUNTIME_PERSISTENT_WORKERS", "1") != "0"
)


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def seed_everything(seed: int, deterministic: bool = CFG.strict_determinism):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = not deterministic
    torch.backends.cudnn.deterministic = deterministic
    try:
        torch.use_deterministic_algorithms(deterministic, warn_only=True)
    except Exception:
        pass


def stable_hash(payload) -> str:
    encoded = json.dumps(payload, sort_keys=True, default=str, separators=(",", ":")).encode()
    return hashlib.sha256(encoded).hexdigest()


def file_sha256(path: str | Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


seed_everything(CFG.seeds[0])
torch.set_float32_matmul_precision("high")
print("Iteration directory:", ITERATION_DIR)
print("Run directory:", RUN_ROOT)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print(
    "Runtime DataLoader:",
    f"workers={RUNTIME_NUM_WORKERS}, "
    f"prefetch_factor={RUNTIME_PREFETCH_FACTOR if RUNTIME_NUM_WORKERS > 0 else 'n/a'}, "
    f"persistent_workers={RUNTIME_PERSISTENT_WORKERS}",
)
print(json.dumps(asdict(CFG), indent=2, default=str))


### Race-safe checkpoint writes
This version uses unique temporary files for JSON/CSV/PyTorch/hash writes before `os.replace`. This prevents collisions such as `run_state.json.tmp` disappearing when more than one writer touches the same directory. **The experiment `Config`, hashes, model, optimizer, seeds, batch size, and training protocol are unchanged.**


## 2 — Locked manifests and dataset audit

`sample_id` is a stable hash of the resolved clean/noisy pair and SNR; the original repeated per-split ID is retained as `source_id`. Speaker and noise overlap are recorded rather than treated as fatal because the fixed repository splits intentionally contain the same speaker and noise label sets.


In [ ]:
REQUIRED_COLUMNS = {
    "id", "speaker_id", "clean_path", "noisy_path", "noise_category", "snr_db", "duration_sec"
}


def resolve_audio_path(value: str, csv_path: Path, dataset_root: Path) -> str:
    path = Path(str(value)).expanduser()
    if path.is_absolute():
        return str(path.resolve())
    candidates = [csv_path.parent / path, dataset_root / path, ITERATION_DIR / path, Path.cwd() / path]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate.resolve())
    return str((ITERATION_DIR / path).resolve())


def load_split_metadata(split: str, cfg: Config = CFG) -> pd.DataFrame:
    root = Path(cfg.dataset_root)
    csv_path = root / split / "metadata.csv"
    frame = pd.read_csv(csv_path)
    missing = REQUIRED_COLUMNS - set(frame.columns)
    if missing:
        raise ValueError(f"{csv_path} is missing columns: {sorted(missing)}")
    frame = frame.copy()
    frame["source_id"] = frame["id"].astype(str)
    frame["split"] = split
    for column in ("clean_path", "noisy_path"):
        frame[column] = frame[column].map(lambda value: resolve_audio_path(value, csv_path, root))
    frame["sample_id"] = [
        hashlib.sha1(f"{clean}|{noisy}|{snr}".encode()).hexdigest()[:20]
        for clean, noisy, snr in zip(frame.clean_path, frame.noisy_path, frame.snr_db)
    ]
    return frame


def _audio_header(path: str):
    try:
        info = sf.info(path)
        return {"path": path, "exists": True, "sample_rate": info.samplerate,
                "frames": info.frames, "duration": info.duration, "error": None}
    except Exception as error:
        return {"path": path, "exists": Path(path).exists(), "sample_rate": None,
                "frames": None, "duration": None, "error": repr(error)}


def audio_header_audit(frames: dict[str, pd.DataFrame], cfg: Config = CFG):
    paths = sorted(set(pd.concat([
        frame[["clean_path", "noisy_path"]].stack() for frame in frames.values()
    ]).astype(str)))
    if not cfg.full_audio_header_audit:
        rng = np.random.default_rng(2026)
        paths = list(rng.choice(paths, size=min(1024, len(paths)), replace=False))
    with ThreadPoolExecutor(max_workers=cfg.audio_audit_workers) as pool:
        rows = list(tqdm(pool.map(_audio_header, paths), total=len(paths), desc="Audio headers"))
    report = pd.DataFrame(rows)
    failures = report[report.error.notna()]
    wrong_sr = report[report.sample_rate.notna() & (report.sample_rate != cfg.sample_rate)]
    empty = report[report.frames.notna() & (report.frames <= 0)]
    if len(failures) or len(wrong_sr) or len(empty):
        raise RuntimeError(
            f"Audio audit failed: unreadable={len(failures)}, wrong_sr={len(wrong_sr)}, empty={len(empty)}"
        )
    return report


def overlap_count(left: pd.Series, right: pd.Series):
    return len(set(left.astype(str)) & set(right.astype(str)))


def audit_splits(frames: dict[str, pd.DataFrame], cfg: Config = CFG):
    expected = {"train": 15_000, "val": 2_000, "test": 2_000}
    rows = []
    for split, frame in frames.items():
        if len(frame) != expected[split]:
            raise ValueError(f"Expected {expected[split]} {split} rows, found {len(frame)}")
        if frame.sample_id.duplicated().any():
            raise ValueError(f"Duplicate stable sample IDs inside {split}")
        if frame.snr_db.isna().any() or not set(frame.snr_db.unique()).issubset(set(cfg.snr_levels)):
            raise ValueError(f"Invalid SNR labels in {split}")
        if (frame.duration_sec <= 0).any():
            raise ValueError(f"Non-positive metadata duration in {split}")
        missing_clean = sum(not Path(path).exists() for path in frame.clean_path)
        missing_noisy = sum(not Path(path).exists() for path in frame.noisy_path)
        if missing_clean or missing_noisy:
            raise FileNotFoundError(f"{split}: clean missing={missing_clean}; noisy missing={missing_noisy}")

    for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
        row = {
            "split_a": left,
            "split_b": right,
            "sample_id_overlap": overlap_count(frames[left].sample_id, frames[right].sample_id),
            "clean_path_overlap": overlap_count(frames[left].clean_path, frames[right].clean_path),
            "noisy_path_overlap": overlap_count(frames[left].noisy_path, frames[right].noisy_path),
            "speaker_overlap": overlap_count(frames[left].speaker_id, frames[right].speaker_id),
            "noise_category_overlap": overlap_count(frames[left].noise_category, frames[right].noise_category),
        }
        if row["sample_id_overlap"] or row["clean_path_overlap"] or row["noisy_path_overlap"]:
            raise RuntimeError(f"Pair/path leakage detected: {row}")
        rows.append(row)
    overlap = pd.DataFrame(rows)
    return {
        "created_at": utc_now(),
        "split_counts": {name: len(frame) for name, frame in frames.items()},
        "unique_speakers": {name: int(frame.speaker_id.nunique()) for name, frame in frames.items()},
        "unique_noise_categories": {name: int(frame.noise_category.nunique()) for name, frame in frames.items()},
        "pairwise_overlap": overlap.to_dict("records"),
        "speaker_independent": False,
        "unseen_noise_main_test": False,
        "limitation": "All 23 speakers and all 11 noise labels occur in each fixed split.",
    }


def _unique_atomic_temp_path(path: str | Path) -> Path:
    """Create a unique temp file in the target directory for race-safe os.replace()."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(
        prefix=f".{path.name}.{os.getpid()}.",
        suffix=".tmp",
        dir=str(path.parent),
    )
    os.close(fd)
    return Path(temporary)


def _cleanup_atomic_temp(temporary: Path):
    try:
        temporary.unlink(missing_ok=True)
    except Exception:
        pass


def atomic_text_save(text: str, path: str | Path):
    path = Path(path)
    temporary = _unique_atomic_temp_path(path)
    try:
        with open(temporary, "w") as handle:
            handle.write(str(text))
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        _cleanup_atomic_temp(temporary)


def atomic_json_save(payload, path: str | Path):
    path = Path(path)
    temporary = _unique_atomic_temp_path(path)
    try:
        with open(temporary, "w") as handle:
            json.dump(payload, handle, indent=2, default=str)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        _cleanup_atomic_temp(temporary)


def atomic_csv_save(frame: pd.DataFrame, path: str | Path):
    path = Path(path)
    temporary = _unique_atomic_temp_path(path)
    try:
        with open(temporary, "w", newline="") as handle:
            frame.to_csv(handle, index=False)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        _cleanup_atomic_temp(temporary)


def prepare_locked_manifests(cfg: Config = CFG, run_audio_audit: bool = True):
    frames = {split: load_split_metadata(split, cfg) for split in ("train", "val", "test")}
    audit = audit_splits(frames, cfg)
    manifest_dir = Path(cfg.run_root) / "manifests"
    manifest_dir.mkdir(parents=True, exist_ok=True)
    hashes = {}
    names = {"train": "train_manifest.csv", "val": "validation_manifest.csv", "test": "test_manifest.csv"}
    for split, frame in frames.items():
        output = manifest_dir / names[split]
        atomic_csv_save(frame.sort_values("sample_id"), output)
        hashes[names[split]] = file_sha256(output)
    atomic_json_save(hashes, manifest_dir / "manifest_hashes.json")
    atomic_json_save(audit, manifest_dir / "split_audit.json")
    if run_audio_audit:
        headers = audio_header_audit(frames, cfg)
        atomic_csv_save(headers, manifest_dir / "audio_header_audit.csv")
    print(json.dumps(audit, indent=2))
    return frames, audit


# Run once before any screening or training:
SPLIT_FRAMES, SPLIT_AUDIT = prepare_locked_manifests()


## 3 — Deterministic audio dataset and resumable SNR-balanced batches


### Runtime-only single-GPU throughput tuning

This version preserves the existing experiment configuration and checkpoint hashes. `Config.batch_size` and `Config.num_workers` are intentionally unchanged. The training/validation DataLoaders use runtime-only worker, prefetch, and persistent-worker settings to improve CPU/audio pipeline throughput on the existing RTX 6000 run without changing the model, optimizer, seeds, epochs, batch composition, or saved checkpoint configuration.


In [ ]:
def load_audio_mono(path: str, target_sr: int = CFG.sample_rate) -> np.ndarray:
    waveform, sample_rate = sf.read(path, dtype="float32")
    if waveform.ndim == 2:
        waveform = waveform.mean(axis=1)
    if sample_rate != target_sr:
        if librosa is not None:
            waveform = librosa.resample(waveform, orig_sr=sample_rate, target_sr=target_sr)
        else:
            from scipy.signal import resample_poly
            divisor = math.gcd(int(sample_rate), int(target_sr))
            waveform = resample_poly(waveform, target_sr // divisor, sample_rate // divisor)
    return np.asarray(waveform, dtype=np.float32)


def deterministic_sample_seed(seed: int, stage: str, epoch: int, sample_id: str) -> int:
    digest = hashlib.sha256(f"{seed}|{stage}|{epoch}|{sample_id}".encode()).digest()
    return int.from_bytes(digest[:8], "little") % (2**32)


def aligned_crop_or_pad(noisy, clean, crop_length, rng=None):
    length = max(len(noisy), len(clean))
    noisy = np.pad(noisy, (0, length - len(noisy)))
    clean = np.pad(clean, (0, length - len(clean)))
    if length >= crop_length:
        start = int(rng.integers(0, length - crop_length + 1)) if rng is not None else (length - crop_length) // 2
        noisy, clean = noisy[start:start + crop_length], clean[start:start + crop_length]
    else:
        noisy = np.pad(noisy, (0, crop_length - length))
        clean = np.pad(clean, (0, crop_length - length))
    return noisy.astype(np.float32), clean.astype(np.float32)


class SpeechDataset(Dataset):
    def __init__(self, frame, cfg: Config = CFG, augment=False, seed=0, stage="validation"):
        self.frame = frame.reset_index(drop=True).copy()
        self.cfg = cfg
        self.augment = augment
        self.seed = int(seed)
        self.stage = str(stage)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, request):
        if isinstance(request, (tuple, list)):
            index, epoch = int(request[0]), int(request[1])
        else:
            index, epoch = int(request), 0
        row = self.frame.iloc[index]
        rng = np.random.default_rng(deterministic_sample_seed(self.seed, self.stage, epoch, row.sample_id))
        noisy = load_audio_mono(row.noisy_path, self.cfg.sample_rate)
        clean = load_audio_mono(row.clean_path, self.cfg.sample_rate)
        noisy, clean = aligned_crop_or_pad(
            noisy, clean, self.cfg.crop_length, rng if self.augment else None
        )
        if self.augment:
            gain = float(rng.uniform(0.80, 1.20))
            polarity = -1.0 if rng.random() < 0.5 else 1.0
            noisy, clean = noisy * gain * polarity, clean * gain * polarity
        peak = max(float(np.max(np.abs(noisy))), float(np.max(np.abs(clean))), 1.0)
        return {
            "noisy": torch.from_numpy(np.clip(noisy / peak, -1, 1).astype(np.float32)),
            "clean": torch.from_numpy(np.clip(clean / peak, -1, 1).astype(np.float32)),
            "snr_db": torch.tensor(float(row.snr_db), dtype=torch.float32),
            "sample_id": str(row.sample_id),
        }


class ResumableBalancedBatchSampler(Sampler):
    # Deterministic epoch batches whose cursor can be checkpointed exactly.
    def __init__(self, frame, batch_size, seed, balance_snr=True, drop_last=False):
        self.frame = frame.reset_index(drop=True)
        self.batch_size = int(batch_size)
        self.seed = int(seed)
        self.balance_snr = bool(balance_snr)
        self.drop_last = bool(drop_last)
        self.epoch = 1
        self.cursor = 0
        self.batches = []
        self._build_epoch()

    def _build_epoch(self):
        generator = torch.Generator().manual_seed(self.seed + 1_000_003 * self.epoch)
        count = len(self.frame)
        if self.balance_snr:
            counts = self.frame.snr_db.value_counts()
            weights = torch.tensor([1.0 / counts.loc[value] for value in self.frame.snr_db], dtype=torch.double)
            indices = torch.multinomial(weights, count, replacement=True, generator=generator).tolist()
        else:
            indices = torch.randperm(count, generator=generator).tolist()
        self.batches = [indices[i:i + self.batch_size] for i in range(0, count, self.batch_size)]
        if self.drop_last and self.batches and len(self.batches[-1]) < self.batch_size:
            self.batches.pop()
        self.cursor = min(self.cursor, len(self.batches))

    def set_epoch(self, epoch, cursor=0):
        self.epoch, self.cursor = int(epoch), int(cursor)
        self._build_epoch()
        self.cursor = int(cursor)

    def __iter__(self):
        # DataLoader prefetch must not advance the completed-batch cursor.
        for position in range(self.cursor, len(self.batches)):
            yield [(index, self.epoch) for index in self.batches[position]]

    def mark_batch_complete(self):
        if self.cursor >= len(self.batches):
            raise RuntimeError("Sampler cursor advanced beyond the epoch")
        self.cursor += 1

    def __len__(self):
        return max(0, len(self.batches) - self.cursor)

    def state_dict(self):
        return {"epoch": self.epoch, "cursor": self.cursor, "seed": self.seed,
                "batch_size": self.batch_size, "balance_snr": self.balance_snr,
                "drop_last": self.drop_last}

    def load_state_dict(self, state):
        for key in ("seed", "batch_size", "balance_snr", "drop_last"):
            if state[key] != getattr(self, key):
                raise ValueError(f"Sampler mismatch for {key}: {state[key]} != {getattr(self, key)}")
        self.set_epoch(state["epoch"], state["cursor"])


def _runtime_loader_kwargs(cfg: Config = CFG):
    """Performance-only DataLoader kwargs; intentionally excluded from Config/hash."""
    kwargs = {
        "num_workers": RUNTIME_NUM_WORKERS,
        "pin_memory": bool(cfg.pin_memory and DEVICE.type == "cuda"),
        "persistent_workers": RUNTIME_PERSISTENT_WORKERS,
    }
    # PyTorch only accepts prefetch_factor when multiprocessing workers are used.
    if RUNTIME_NUM_WORKERS > 0:
        kwargs["prefetch_factor"] = RUNTIME_PREFETCH_FACTOR
    return kwargs


def make_train_loader(frame, seed, stage, cfg: Config = CFG):
    dataset = SpeechDataset(frame, cfg, augment=True, seed=seed, stage=stage)
    sampler = ResumableBalancedBatchSampler(
        frame,
        cfg.batch_size,
        seed,
        balance_snr=True,
    )
    loader = DataLoader(
        dataset,
        batch_sampler=sampler,
        **_runtime_loader_kwargs(cfg),
    )
    return loader, sampler


def make_validation_loader(frame, seed, cfg: Config = CFG):
    dataset = SpeechDataset(frame, cfg, augment=False, seed=seed, stage="validation")
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        **_runtime_loader_kwargs(cfg),
    )


## 4 — Complex STFT frontend and exact-length reconstruction


In [ ]:
class STFTProcessor(nn.Module):
    def __init__(self, cfg: Config = CFG):
        super().__init__()
        self.cfg = cfg
        self.register_buffer("window", torch.hann_window(cfg.win_length), persistent=False)

    def forward(self, waveform):
        if waveform.ndim == 1:
            waveform = waveform.unsqueeze(0)
        spectrum = torch.stft(
            waveform.float(), n_fft=self.cfg.n_fft, hop_length=self.cfg.hop_length,
            win_length=self.cfg.win_length, window=self.window.float(), center=True,
            pad_mode="reflect", return_complex=True,
        )
        return torch.stack([spectrum.real, spectrum.imag], dim=1).permute(0, 1, 3, 2).contiguous()

    def inverse(self, spectrum_ri, length):
        real = spectrum_ri[:, 0].permute(0, 2, 1).float().contiguous()
        imag = spectrum_ri[:, 1].permute(0, 2, 1).float().contiguous()
        return torch.istft(torch.complex(real, imag), n_fft=self.cfg.n_fft,
                           hop_length=self.cfg.hop_length, win_length=self.cfg.win_length,
                           window=self.window.float(), center=True, length=int(length))


def spectrum_features(spectrum_ri, eps=1e-8):
    real, imag = spectrum_ri[:, 0:1], spectrum_ri[:, 1:2]
    magnitude = torch.sqrt(real.square() + imag.square() + eps)
    scale = magnitude.detach().amax(dim=(-2, -1), keepdim=True).clamp_min(eps)
    log_scale = torch.log1p(scale).clamp_min(eps)
    return torch.cat([real / scale, imag / scale, torch.log1p(magnitude) / log_scale,
                      imag / magnitude.clamp_min(eps), real / magnitude.clamp_min(eps)], dim=1)


def match_spatial(tensor, reference):
    return F.interpolate(tensor, size=reference.shape[-2:], mode="bilinear", align_corners=False)


## 5 — Exact state-vector VQC and exactly matched MLP


In [ ]:
class ExactStatevectorVQC(nn.Module):
    # Batched differentiable six-qubit pure-state simulator in PyTorch.
    def __init__(self, n_qubits=6, depth=2, entanglement="ring", data_reuploading=True):
        super().__init__()
        if n_qubits != 6:
            raise ValueError("This registered protocol uses exactly six qubits")
        if depth not in {2, 4}:
            raise ValueError("depth must be 2 or 4")
        if entanglement not in {"ring", "alternating_ring"}:
            raise ValueError("Unsupported entanglement")
        self.n_qubits, self.depth = n_qubits, depth
        self.entanglement, self.data_reuploading = entanglement, data_reuploading
        self.theta = nn.Parameter(0.05 * torch.randn(depth, n_qubits, 3))
        dimension = 2 ** n_qubits
        z = []
        for wire in range(n_qubits):
            shift = n_qubits - 1 - wire
            z.append([1.0 if ((basis >> shift) & 1) == 0 else -1.0 for basis in range(dimension)])
        z = torch.tensor(z, dtype=torch.float32)
        self.register_buffer("z_signs", z, persistent=False)
        self.register_buffer("zz_signs", z * torch.roll(z, -1, 0), persistent=False)
        self.runtime_shots = None
        self.runtime_readout_error = 0.0
        self.runtime_angle_noise_std = 0.0
        self.runtime_depolarizing = 0.0

    @property
    def output_dim(self):
        return 12

    def set_runtime_noise(self, shots=None, readout_error=0.0, angle_noise_std=0.0, depolarizing=0.0):
        if shots is not None and shots <= 0:
            raise ValueError("shots must be positive or None")
        if not 0 <= readout_error < 0.5 or angle_noise_std < 0 or not 0 <= depolarizing < 1:
            raise ValueError("Invalid quantum runtime noise")
        self.runtime_shots = shots
        self.runtime_readout_error = float(readout_error)
        self.runtime_angle_noise_std = float(angle_noise_std)
        self.runtime_depolarizing = float(depolarizing)

    @staticmethod
    def _ry(angle):
        c, s = torch.cos(angle / 2), torch.sin(angle / 2)
        return torch.stack([torch.stack([c, -s], -1), torch.stack([s, c], -1)], -2).to(torch.complex64)

    @staticmethod
    def _rz(angle):
        zero = torch.zeros_like(angle)
        a = torch.complex(torch.cos(-angle / 2), torch.sin(-angle / 2))
        b = torch.complex(torch.cos(angle / 2), torch.sin(angle / 2))
        return torch.stack([torch.stack([a, torch.complex(zero, zero)], -1),
                            torch.stack([torch.complex(zero, zero), b], -1)], -2)

    def _single(self, state, gate, wire):
        batch, axis = state.shape[0], wire + 1
        tensor = state.reshape(batch, *([2] * self.n_qubits))
        permutation = [0] + [value for value in range(1, self.n_qubits + 1) if value != axis] + [axis]
        inverse = [0] * len(permutation)
        for index, value in enumerate(permutation):
            inverse[value] = index
        tensor = tensor.permute(permutation).reshape(batch, -1, 2)
        a0, a1 = tensor[..., 0], tensor[..., 1]
        if gate.ndim == 2:
            o0, o1 = a0 * gate[0, 0] + a1 * gate[0, 1], a0 * gate[1, 0] + a1 * gate[1, 1]
        else:
            o0 = a0 * gate[:, None, 0, 0] + a1 * gate[:, None, 0, 1]
            o1 = a0 * gate[:, None, 1, 0] + a1 * gate[:, None, 1, 1]
        return torch.stack([o0, o1], -1).reshape(batch, *([2] * self.n_qubits)).permute(inverse).reshape(batch, -1)

    def _cnot(self, state, control, target):
        dimension = 2 ** self.n_qubits
        permutation = []
        for basis in range(dimension):
            bit = (basis >> (self.n_qubits - 1 - control)) & 1
            permutation.append(basis ^ (1 << (self.n_qubits - 1 - target)) if bit else basis)
        return state[:, torch.tensor(permutation, device=state.device)]

    def _edges(self, layer):
        if self.entanglement == "ring" or layer % 2 == 0:
            return [(wire, (wire + 1) % self.n_qubits) for wire in range(self.n_qubits)]
        return [((wire + 1) % self.n_qubits, wire) for wire in range(self.n_qubits)]

    def forward(self, inputs, return_state=False):
        if inputs.shape[-1] != self.n_qubits:
            raise ValueError(f"Expected six inputs, received {inputs.shape}")
        inputs = inputs.float()
        batch = inputs.shape[0]
        state = torch.zeros(batch, 2 ** self.n_qubits, dtype=torch.complex64, device=inputs.device)
        state[:, 0] = 1.0 + 0.0j
        for layer in range(self.depth):
            if layer == 0 or self.data_reuploading:
                encoded = inputs
                if self.training and self.runtime_angle_noise_std:
                    encoded = encoded + self.runtime_angle_noise_std * torch.randn_like(encoded)
                for wire in range(self.n_qubits):
                    state = self._single(state, self._ry(encoded[:, wire]), wire)
            for wire in range(self.n_qubits):
                for gate_fn, parameter in zip((self._rz, self._ry, self._rz), self.theta[layer, wire]):
                    state = self._single(state, gate_fn(parameter), wire)
            for control, target in self._edges(layer):
                state = self._cnot(state, control, target)
        probabilities = state.abs().square()
        expectations = torch.cat([probabilities @ self.z_signs.T,
                                  probabilities @ self.zz_signs.T], dim=-1)
        expectations = expectations * (1.0 - 2.0 * self.runtime_readout_error)
        if self.runtime_depolarizing:
            expectations = expectations * (1.0 - self.runtime_depolarizing) ** self.depth
        if self.runtime_shots is not None:
            plus = ((expectations + 1) / 2).clamp(0, 1)
            counts = torch.distributions.Binomial(float(self.runtime_shots), probs=plus).sample()
            expectations = 2 * counts / float(self.runtime_shots) - 1
        return (expectations, state) if return_state else expectations


class ParameterMatchedMLP(nn.Module):
    def __init__(self, n_qubits=6, depth=2):
        super().__init__()
        hidden = 3 if depth == 2 else 6 if depth == 4 else None
        if hidden is None:
            raise ValueError("depth must be 2 or 4")
        self.net = nn.Sequential(nn.Linear(n_qubits, hidden, bias=False), nn.Tanh(),
                                 nn.Linear(hidden, n_qubits, bias=False), nn.Tanh())

    @property
    def output_dim(self):
        return 12

    def forward(self, inputs):
        values = self.net(inputs)
        return torch.cat([values, values * torch.roll(values, -1, -1)], dim=-1)


## 6 — Multi-location conditioned lightweight U-Net


In [ ]:
class ResidualDSGLU(nn.Module):
    def __init__(self, channels, dilation=1, dropout=CFG.dropout):
        super().__init__()
        self.norm = nn.GroupNorm(1, channels)
        self.depthwise = nn.Conv2d(channels, channels, 3, padding=dilation,
                                   dilation=dilation, groups=channels, bias=False)
        self.pointwise = nn.Conv2d(channels, 2 * channels, 1)
        self.dropout = nn.Dropout2d(dropout)

    def forward(self, inputs):
        value, gate = self.pointwise(self.depthwise(self.norm(inputs))).chunk(2, dim=1)
        return inputs + self.dropout(value * torch.sigmoid(gate))


class EncoderScale(nn.Module):
    def __init__(self, channels, blocks, dropout):
        super().__init__()
        self.blocks = nn.Sequential(*[ResidualDSGLU(channels, dilation=1, dropout=dropout)
                                      for _ in range(blocks)])

    def forward(self, inputs):
        return self.blocks(inputs)


class Downsample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(in_channels, out_channels, 3, stride=2, padding=1, bias=False),
                                 nn.GroupNorm(1, out_channels), nn.SiLU())

    def forward(self, inputs):
        return self.net(inputs)


class DecoderScale(nn.Module):
    def __init__(self, in_channels, skip_channels, blocks, dropout):
        super().__init__()
        self.fuse = nn.Sequential(nn.Conv2d(in_channels + skip_channels, skip_channels, 1, bias=False),
                                  nn.GroupNorm(1, skip_channels), nn.SiLU())
        self.blocks = nn.Sequential(*[ResidualDSGLU(skip_channels, dropout=dropout) for _ in range(blocks)])

    def forward(self, inputs, skip):
        inputs = match_spatial(inputs, skip)
        return self.blocks(self.fuse(torch.cat([inputs, skip], dim=1)))


class GlobalFeatureConditioner(nn.Module):
    def __init__(self, target_channels, statistic_channels, variant, cfg: Config):
        super().__init__()
        if variant not in {"classical", "qnn"}:
            raise ValueError("Conditioners exist only for classical or qnn")
        self.variant = variant
        self.input_projection = nn.Sequential(nn.Linear(2 * statistic_channels, cfg.n_qubits),
                                              nn.LayerNorm(cfg.n_qubits))
        self.core = (ExactStatevectorVQC(cfg.n_qubits, cfg.q_depth, cfg.entanglement,
                                         cfg.data_reuploading) if variant == "qnn"
                     else ParameterMatchedMLP(cfg.n_qubits, cfg.q_depth))
        self.output_projection = nn.Linear(12, 2 * target_channels)
        nn.init.normal_(self.output_projection.weight, mean=0.0, std=1e-3)
        nn.init.zeros_(self.output_projection.bias)
        self.logit_strength = nn.Parameter(torch.tensor(-2.5))

    def set_quantum_runtime(self, **kwargs):
        if self.variant == "qnn":
            self.core.set_runtime_noise(**kwargs)

    def forward(self, target, statistic_tensors, return_aux=False):
        statistics = []
        for tensor in statistic_tensors:
            statistics.extend([tensor.mean(dim=(-2, -1)), tensor.std(dim=(-2, -1), unbiased=False)])
        angles = math.pi * torch.tanh(self.input_projection(torch.cat(statistics, dim=-1)))
        if self.variant == "qnn":
            with torch.autocast(device_type=target.device.type, enabled=False):
                latent = self.core(angles.float())
        else:
            latent = self.core(angles)
        scale, bias = self.output_projection(latent).chunk(2, dim=-1)
        scale = torch.tanh(scale).unsqueeze(-1).unsqueeze(-1)
        bias = torch.tanh(bias).unsqueeze(-1).unsqueeze(-1)
        strength = 0.25 * torch.sigmoid(self.logit_strength)
        conditioned = target * (1.0 + strength * scale) + strength * bias
        if return_aux:
            return conditioned, {"angles": angles.detach(), "latent": latent.detach(),
                                 "strength": strength.detach(), "scale": scale.detach(), "bias": bias.detach()}
        return conditioned


class QuantumUNet(nn.Module):
    def __init__(self, variant="qnn", config: Config = CFG):
        super().__init__()
        if variant not in {"baseline", "classical", "qnn"}:
            raise ValueError("variant must be baseline, classical, or qnn")
        if config.conditioning_layout not in {"bottleneck_only", "bottleneck_and_two_skip_gates"}:
            raise ValueError("Unsupported conditioning layout")
        self.variant, self.cfg = variant, config
        channels = list(config.encoder_channels)
        self.stft = STFTProcessor(config)
        self.stem = nn.Sequential(nn.Conv2d(5, channels[0], 3, padding=1, bias=False),
                                  nn.GroupNorm(1, channels[0]), nn.SiLU())
        self.encoders = nn.ModuleList([EncoderScale(c, config.blocks_per_scale, config.dropout)
                                       for c in channels])
        self.downsamples = nn.ModuleList([Downsample(channels[i], channels[i + 1])
                                          for i in range(len(channels) - 1)])
        dilations = (1, 2, 4)[:config.bottleneck_blocks]
        self.bottleneck = nn.Sequential(*[ResidualDSGLU(channels[-1], dilation=d, dropout=config.dropout)
                                          for d in dilations])
        self.decoders = nn.ModuleList([
            DecoderScale(channels[i + 1], channels[i], config.blocks_per_scale, config.dropout)
            for i in reversed(range(len(channels) - 1))
        ])
        self.head = nn.Sequential(ResidualDSGLU(channels[0], dropout=config.dropout),
                                  nn.Conv2d(channels[0], 4, 1))
        self.conditioners = nn.ModuleDict()
        if variant != "baseline":
            # Bottleneck statistics include four encoder scales plus processed bottleneck.
            self.conditioners["bottleneck"] = GlobalFeatureConditioner(
                channels[-1], sum(channels) + channels[-1], variant, config
            )
            if config.conditioning_layout == "bottleneck_and_two_skip_gates":
                self.conditioners["skip_deep"] = GlobalFeatureConditioner(
                    channels[-2], channels[-2] + channels[-1], variant, config
                )
                self.conditioners["skip_mid"] = GlobalFeatureConditioner(
                    channels[-3], channels[-3] + channels[-1], variant, config
                )

    @property
    def quantum_calls_per_segment(self):
        return len(self.conditioners) if self.variant == "qnn" else 0

    def set_quantum_runtime(self, shots=None, readout_error=0.0, angle_noise_std=0.0, depolarizing=0.0):
        for conditioner in self.conditioners.values():
            conditioner.set_quantum_runtime(shots=shots, readout_error=readout_error,
                                            angle_noise_std=angle_noise_std, depolarizing=depolarizing)

    def forward(self, noisy_waveform, return_aux=False):
        original_length = noisy_waveform.shape[-1]
        noisy_stft = self.stft(noisy_waveform.float())
        hidden = self.stem(spectrum_features(noisy_stft))
        skips = []
        for index, encoder in enumerate(self.encoders):
            hidden = encoder(hidden)
            skips.append(hidden)
            if index < len(self.downsamples):
                hidden = self.downsamples[index](hidden)
        hidden = self.bottleneck(hidden)
        bottleneck_context = hidden
        auxiliary = {}
        if self.variant != "baseline":
            if return_aux:
                hidden, auxiliary["bottleneck"] = self.conditioners["bottleneck"](
                    hidden, skips + [hidden], return_aux=True
                )
            else:
                hidden = self.conditioners["bottleneck"](hidden, skips + [hidden])

        # Decoder consumes encoder scales 96, 64, 32 in that order.
        for decoder_index, (decoder, skip_index) in enumerate(zip(self.decoders, (2, 1, 0))):
            skip = skips[skip_index]
            key = "skip_deep" if skip_index == 2 else "skip_mid" if skip_index == 1 else None
            if key in self.conditioners:
                if return_aux:
                    skip, auxiliary[key] = self.conditioners[key](
                        skip, [skip, bottleneck_context], return_aux=True
                    )
                else:
                    skip = self.conditioners[key](skip, [skip, bottleneck_context])
            hidden = decoder(hidden, skip)

        prediction = self.head(hidden)
        prediction = match_spatial(prediction, noisy_stft)
        magnitude_mask = 2.0 * torch.sigmoid(prediction[:, 0:1])
        phase_residual = self.cfg.phase_residual_scale * torch.tanh(prediction[:, 1:2])
        real_residual = self.cfg.complex_residual_scale * torch.tanh(prediction[:, 2:3])
        imag_residual = self.cfg.complex_residual_scale * torch.tanh(prediction[:, 3:4])
        noisy_real, noisy_imag = noisy_stft[:, 0:1], noisy_stft[:, 1:2]
        magnitude = torch.sqrt(noisy_real.square() + noisy_imag.square() + 1e-8)
        phase = torch.atan2(noisy_imag, noisy_real)
        enhanced_magnitude = magnitude * magnitude_mask
        enhanced_real = enhanced_magnitude * torch.cos(phase + phase_residual) + real_residual * magnitude
        enhanced_imag = enhanced_magnitude * torch.sin(phase + phase_residual) + imag_residual * magnitude
        enhanced_stft = torch.cat([enhanced_real, enhanced_imag], dim=1)
        enhanced = self.stft.inverse(enhanced_stft, original_length)
        if return_aux:
            auxiliary.update({"magnitude_mask": magnitude_mask.detach(),
                              "phase_residual": phase_residual.detach()})
            return enhanced, enhanced_stft, auxiliary
        return enhanced, enhanced_stft

    @torch.inference_mode()
    def enhance(self, waveform):
        self.eval()
        return self.forward(waveform)[0]


## 7 — Parameter equality, resource reporting, and verification tests


In [ ]:
def count_trainable(module):
    return sum(parameter.numel() for parameter in module.parameters() if parameter.requires_grad)


def tensor_dict_hash(state_dict, include=None, exclude=None):
    digest = hashlib.sha256()
    for name in sorted(state_dict):
        if include and not include(name):
            continue
        if exclude and exclude(name):
            continue
        tensor = state_dict[name].detach().cpu().contiguous()
        digest.update(name.encode())
        digest.update(str(tuple(tensor.shape)).encode())
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()


def backbone_hash(model):
    return tensor_dict_hash(model.state_dict(), exclude=lambda name: name.startswith("conditioners."))


def conditioner_shell_hash(model):
    return tensor_dict_hash(
        model.state_dict(), include=lambda name: name.startswith("conditioners."),
        exclude=lambda name: ".core." in name
    )


def core_parameters(model):
    return sum(count_trainable(conditioner.core) for conditioner in model.conditioners.values())


def conditioner_parameters(model):
    return count_trainable(model.conditioners)


def reset_conditioner_shell(model, seed):
    # Initialize every non-core conditioner tensor identically across matched arms.
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(int(seed) + 91_337)
        for conditioner in model.conditioners.values():
            linear = conditioner.input_projection[0]
            nn.init.xavier_uniform_(linear.weight)
            nn.init.zeros_(linear.bias)
            nn.init.ones_(conditioner.input_projection[1].weight)
            nn.init.zeros_(conditioner.input_projection[1].bias)
            nn.init.normal_(conditioner.output_projection.weight, mean=0.0, std=1e-3)
            nn.init.zeros_(conditioner.output_projection.bias)
            conditioner.logit_strength.data.fill_(-2.5)
    return model


def build_matched_pair(cfg: Config, seed: int, baseline_checkpoint=None):
    seed_everything(seed)
    classical = reset_conditioner_shell(QuantumUNet("classical", cfg), seed)
    seed_everything(seed)
    qnn = reset_conditioner_shell(QuantumUNet("qnn", cfg), seed)
    if baseline_checkpoint is not None:
        checkpoint = safe_torch_load(baseline_checkpoint, map_location="cpu")
        baseline_state = checkpoint["model_state"]
        classical.load_state_dict(baseline_state, strict=False)
        qnn.load_state_dict(baseline_state, strict=False)
    assert core_parameters(classical) == core_parameters(qnn)
    assert conditioner_parameters(classical) == conditioner_parameters(qnn)
    assert count_trainable(classical) == count_trainable(qnn)
    assert backbone_hash(classical) == backbone_hash(qnn)
    assert conditioner_shell_hash(classical) == conditioner_shell_hash(qnn)
    return classical, qnn


def quantum_resource_report(model):
    circuits = len(model.conditioners) if model.variant == "qnn" else 0
    depth = model.cfg.q_depth if model.variant == "qnn" else 0
    cnot_per_layer = model.cfg.n_qubits if model.variant == "qnn" else 0
    return {
        "variant": model.variant,
        "total_parameters": count_trainable(model),
        "conditioner_parameters": conditioner_parameters(model),
        "core_parameters": core_parameters(model),
        "qubits_per_circuit": model.cfg.n_qubits if model.variant == "qnn" else 0,
        "statevector_dimension": 2 ** model.cfg.n_qubits if model.variant == "qnn" else 0,
        "circuit_depth": depth,
        "circuit_calls_per_segment": circuits,
        "trainable_quantum_parameters": circuits * depth * model.cfg.n_qubits * 3,
        "encoding_gates": circuits * depth * model.cfg.n_qubits,
        "rotation_gates": circuits * depth * model.cfg.n_qubits * 3,
        "cnot_gates": circuits * depth * cnot_per_layer,
    }


def run_quantum_unit_tests(device=DEVICE):
    seed_everything(2026)
    circuit = ExactStatevectorVQC(depth=2, entanglement="ring").to(device)
    inputs = torch.zeros(3, 6, device=device, requires_grad=True)
    output, state = circuit(inputs, return_state=True)
    assert torch.allclose(state.abs().square().sum(-1), torch.ones(3, device=device), atol=2e-5)
    assert output.min() >= -1.00001 and output.max() <= 1.00001
    assert torch.allclose(output, torch.ones_like(output), atol=2e-3) is False  # trainable gates are active
    output.sum().backward()
    assert inputs.grad is not None and torch.isfinite(inputs.grad).all()
    assert circuit.theta.grad is not None and torch.isfinite(circuit.theta.grad).all()
    # |000000> with zero trainable angles has all Z and ZZ expectations equal to +1.
    with torch.no_grad():
        circuit.theta.zero_()
        zero_output = circuit(torch.zeros(1, 6, device=device))
    assert torch.allclose(zero_output, torch.ones_like(zero_output), atol=2e-5)
    mlp = ParameterMatchedMLP(depth=2)
    assert count_trainable(mlp) == count_trainable(circuit) == 36
    print("Quantum unit tests passed.")


def run_model_fairness_smoke_tests(cfg: Config = CFG, device=DEVICE):
    baseline = QuantumUNet("baseline", cfg)
    classical, qnn = build_matched_pair(cfg, seed=2026)
    counts = pd.DataFrame([
        {"variant": model.variant, "total": count_trainable(model),
         "conditioners": conditioner_parameters(model), "cores": core_parameters(model)}
        for model in (baseline, classical, qnn)
    ])
    display(counts)
    assert cfg.min_parameters <= count_trainable(qnn) <= cfg.max_parameters, counts
    assert count_trainable(classical) == count_trainable(qnn)
    length = cfg.sample_rate + 137  # odd time-bin/length stress case
    for model in (baseline, classical, qnn):
        model = model.to(device)
        waveform = torch.randn(1, length, device=device) * 0.02
        enhanced, spectrum = model(waveform)
        assert enhanced.shape == waveform.shape
        assert spectrum.shape[1] == 2
        assert torch.isfinite(enhanced).all() and torch.isfinite(spectrum).all()
        enhanced.square().mean().backward()
        assert all(parameter.grad is None or torch.isfinite(parameter.grad).all()
                   for parameter in model.parameters())
        if model.variant != "baseline":
            gradients = [parameter.grad.abs().sum().item() for name, parameter in model.named_parameters()
                         if ".core." in name and parameter.grad is not None]
            assert gradients and sum(gradients) > 0
        del model, waveform, enhanced, spectrum
        if device.type == "cuda":
            torch.cuda.empty_cache()
    print("Model, shape, gradient, parameter, and fairness smoke tests passed.")
    return counts


def verify_against_pennylane(atol=2e-5):
    try:
        import pennylane as qml
    except ImportError:
        print("PennyLane is optional and not installed; independent verification skipped.")
        return None
    circuit = ExactStatevectorVQC(depth=2, entanglement="ring").cpu()
    values = torch.linspace(-0.5, 0.5, 6).unsqueeze(0)
    dev = qml.device("default.qubit", wires=6)
    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def reference(x, theta):
        for layer in range(2):
            for wire in range(6):
                qml.RY(x[wire], wires=wire)
            for wire in range(6):
                qml.RZ(theta[layer, wire, 0], wires=wire)
                qml.RY(theta[layer, wire, 1], wires=wire)
                qml.RZ(theta[layer, wire, 2], wires=wire)
            for wire in range(6):
                qml.CNOT(wires=[wire, (wire + 1) % 6])
        observables = [qml.PauliZ(wire) for wire in range(6)]
        observables += [qml.PauliZ(wire) @ qml.PauliZ((wire + 1) % 6) for wire in range(6)]
        return [qml.expval(observable) for observable in observables]
    expected = torch.stack(reference(values[0], circuit.theta)).float().unsqueeze(0)
    actual = circuit(values)
    assert torch.allclose(actual, expected, atol=atol), (actual, expected)
    print("Exact simulator agrees with PennyLane.")
    return True


# Required lightweight tests before full training:
run_quantum_unit_tests()
PARAMETER_REPORT = run_model_fairness_smoke_tests()
verify_against_pennylane()  # optional


## 8 — Per-sample composite speech-enhancement loss


In [ ]:
def weighted_mean(values, weights):
    values = values.reshape(values.shape[0], -1).mean(-1) if values.ndim > 1 else values
    return (values * weights).sum() / weights.sum().clamp_min(1e-8)


def snr_weights(snr_db, enabled=True):
    if not enabled:
        return torch.ones_like(snr_db)
    weights = torch.ones_like(snr_db)
    table = {-5: 1.50, 0: 1.30, 5: 1.15, 10: 1.00, 15: 1.00, 20: 1.00}
    for snr, weight in table.items():
        weights = torch.where(snr_db == float(snr), torch.full_like(weights, weight), weights)
    return weights


def si_sdr_per_sample(estimate, reference, eps=1e-8):
    estimate = estimate - estimate.mean(-1, keepdim=True)
    reference = reference - reference.mean(-1, keepdim=True)
    scale = (estimate * reference).sum(-1, keepdim=True) / (reference.square().sum(-1, keepdim=True) + eps)
    target = scale * reference
    noise = estimate - target
    return 10 * torch.log10((target.square().sum(-1) + eps) / (noise.square().sum(-1) + eps))


def mrstft_per_sample(estimate, reference, eps=1e-7):
    losses = torch.zeros(estimate.shape[0], device=estimate.device)
    for n_fft, hop in ((256, 64), (512, 128), (1024, 256)):
        window = torch.hann_window(n_fft, device=estimate.device)
        est = torch.stft(estimate.float(), n_fft=n_fft, hop_length=hop, win_length=n_fft,
                         window=window, center=True, return_complex=True).abs().clamp_min(eps)
        ref = torch.stft(reference.float(), n_fft=n_fft, hop_length=hop, win_length=n_fft,
                         window=window, center=True, return_complex=True).abs().clamp_min(eps)
        convergence = (est - ref).flatten(1).norm(dim=1) / ref.flatten(1).norm(dim=1).clamp_min(eps)
        losses += convergence + (est.log() - ref.log()).abs().flatten(1).mean(1)
    return losses / 3


class CompositeEnhancementLoss(nn.Module):
    def __init__(self, cfg: Config = CFG):
        super().__init__()
        self.cfg = cfg

    def forward(self, enhanced_waveform, enhanced_stft, clean_waveform, clean_stft, snr_db):
        weights = snr_weights(snr_db, self.cfg.low_snr_weighting)
        pred_real, pred_imag = enhanced_stft[:, 0], enhanced_stft[:, 1]
        clean_real, clean_imag = clean_stft[:, 0], clean_stft[:, 1]
        pred_mag = torch.sqrt(pred_real.square() + pred_imag.square() + 1e-8)
        clean_mag = torch.sqrt(clean_real.square() + clean_imag.square() + 1e-8)
        ri = F.smooth_l1_loss(enhanced_stft, clean_stft, reduction="none").flatten(1).mean(1)
        magnitude = F.smooth_l1_loss(pred_mag, clean_mag, reduction="none").flatten(1).mean(1)
        phase_difference = torch.atan2(pred_imag, pred_real) - torch.atan2(clean_imag, clean_real)
        phase_weight = clean_mag / clean_mag.flatten(1).mean(1)[:, None, None].clamp_min(1e-8)
        phase = (phase_weight.clamp(0, 2) * (1 - torch.cos(phase_difference))).flatten(1).mean(1)
        values = {
            "complex": weighted_mean(ri, weights),
            "magnitude": weighted_mean(magnitude, weights),
            "phase": weighted_mean(phase, weights),
            "mrstft": weighted_mean(mrstft_per_sample(enhanced_waveform, clean_waveform), weights),
            "sisdr": weighted_mean(-si_sdr_per_sample(enhanced_waveform, clean_waveform), weights),
        }
        values["total"] = (self.cfg.loss_complex_ri * values["complex"]
                           + self.cfg.loss_magnitude * values["magnitude"]
                           + self.cfg.loss_phase * values["phase"]
                           + self.cfg.loss_mrstft * values["mrstft"]
                           + self.cfg.loss_sisdr * values["sisdr"])
        return values


## 9 — Atomic checkpoints, exact resume state, and history integrity

\`last_step.pt\` always represents a completed optimizer step. The sampler cursor is advanced before yielding a batch, and augmentation is derived from a stable per-sample seed, so a restart continues at the next unprocessed batch without relying on worker RNG timing.


In [ ]:
def atomic_torch_save(payload, path: str | Path):
    path = Path(path)
    temporary = _unique_atomic_temp_path(path)
    try:
        with open(temporary, "wb") as handle:
            torch.save(payload, handle)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        _cleanup_atomic_temp(temporary)


def rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all(state["cuda"])


def safe_torch_load(path, map_location="cpu"):
    """Load full research checkpoints on PyTorch 2.6+ and older releases."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        # Older PyTorch versions do not expose weights_only.
        return torch.load(path, map_location=map_location)


def load_valid_checkpoint(path, config_hash, device="cpu"):
    path = Path(path)
    if not path.exists():
        return None
    try:
        checkpoint = safe_torch_load(path, map_location=device)
        if not isinstance(checkpoint, dict):
            print(f"Ignoring invalid non-dictionary checkpoint: {path}")
            return None
        if checkpoint.get("config_hash") != config_hash:
            print(f"Ignoring checkpoint with mismatched config hash: {path}")
            return None
        return checkpoint
    except Exception as error:
        print(f"Ignoring invalid checkpoint {path}: {error}")
        return None

def clean_history(frame):
    if frame is None or frame.empty:
        return pd.DataFrame()
    required = {"epoch", "stage", "completed_at"}
    if not required.issubset(frame.columns):
        raise ValueError(f"History lacks columns: {required - set(frame.columns)}")
    frame = frame.copy()
    frame["epoch"] = pd.to_numeric(frame.epoch, errors="raise").astype(int)
    return (frame.sort_values(["stage", "epoch", "completed_at"])
            .drop_duplicates(["stage", "epoch"], keep="last")
            .sort_values(["stage", "epoch"]).reset_index(drop=True))


def save_history(frame, path):
    atomic_csv_save(clean_history(frame), path)


class EarlyStopping:
    def __init__(self, patience, min_delta):
        self.patience, self.min_delta = int(patience), float(min_delta)
        self.best, self.bad_epochs, self.best_epoch = float("inf"), 0, 0

    def update(self, value, epoch):
        improved = value < self.best - self.min_delta
        if improved:
            self.best, self.bad_epochs, self.best_epoch = float(value), 0, int(epoch)
        else:
            self.bad_epochs += 1
        return improved, self.bad_epochs >= self.patience

    def state_dict(self):
        return {"patience": self.patience, "min_delta": self.min_delta, "best": self.best,
                "bad_epochs": self.bad_epochs, "best_epoch": self.best_epoch}

    def load_state_dict(self, state):
        self.best, self.bad_epochs, self.best_epoch = state["best"], state["bad_epochs"], state["best_epoch"]


def stage_paths(run_dir):
    root = Path(run_dir)
    names = {"best": "best.pt", "last": "last.pt", "step": "last_step.pt",
             "history": "history.csv", "complete": "complete.json", "config": "config.json",
             "hash": "config_hash.txt", "state": "run_state.json"}
    return {key: root / value for key, value in names.items()}


def write_stage_config(run_dir, cfg, metadata):
    paths = stage_paths(run_dir)
    payload = {"config": asdict(cfg), **metadata}
    config_hash = stable_hash(payload)
    Path(run_dir).mkdir(parents=True, exist_ok=True)
    if paths["hash"].exists():
        existing_hash = paths["hash"].read_text().strip()
        if existing_hash and existing_hash != config_hash:
            raise RuntimeError(f"Refusing to overwrite run with a different configuration: {run_dir}")
    atomic_json_save(payload, paths["config"])
    atomic_text_save(config_hash + "\n", paths["hash"])
    return config_hash, paths


def resume_decision(run_dir, config_hash, maximum_epochs):
    """Choose the safest resume source without repeating completed epochs.

    Priority:
      1) explicit completion marker with a valid best checkpoint
      2) recover completion from history/max epoch or early-stop state
      3) exact mid-epoch checkpoint
      4) exact last completed epoch checkpoint
      5) legacy best checkpoint
      6) new run
    """
    paths = stage_paths(run_dir)

    # 1. Explicit completion is valid only if the best checkpoint is readable.
    if paths["complete"].exists():
        best = load_valid_checkpoint(paths["best"], config_hash)
        if best is not None:
            print("Completed stage: skipping")
            return "complete", None
        print("Completion marker exists but best.pt is not valid; attempting recovery.")

    # Read history once.
    history = pd.DataFrame()
    if paths["history"].exists():
        try:
            history = clean_history(pd.read_csv(paths["history"]))
        except Exception as error:
            print(f"Could not read history for resume decision: {error}")

    # 2a. Recover a max-epoch completion BEFORE last.pt / last_step.pt.
    if not history.empty and int(history.epoch.max()) >= int(maximum_epochs):
        best = load_valid_checkpoint(paths["best"], config_hash)
        if best is not None:
            atomic_json_save({
                "last_completed_epoch": int(history.epoch.max()),
                "best_epoch": int(best.get("epoch", 0)),
                "completion_reason": "recovered_max_epochs",
                "best_validation_loss": float(best.get("validation_loss", np.nan)),
                "configuration_hash": config_hash,
                "completion_timestamp": utc_now(),
            }, paths["complete"])
            print("Completed stage recovered from history: skipping")
            return "complete", None

    # Load latest epoch checkpoint for both completion recovery and resume.
    last = load_valid_checkpoint(paths["last"], config_hash)

    # 2b. Recover early stopping if the kernel died after the final epoch save
    # but before complete.json was written.
    if last is not None:
        early_state = last.get("early_stopping") or {}
        patience = int(early_state.get("patience", 10**9))
        bad_epochs = int(early_state.get("bad_epochs", 0))
        stopped_early = bad_epochs >= patience
        reached_max = int(last.get("epoch", 0)) >= int(maximum_epochs)
        if stopped_early or reached_max:
            best = load_valid_checkpoint(paths["best"], config_hash)
            if best is not None:
                reason = "recovered_early_stopping" if stopped_early else "recovered_max_epochs"
                atomic_json_save({
                    "last_completed_epoch": int(last.get("epoch", 0)),
                    "best_epoch": int(best.get("epoch", 0)),
                    "completion_reason": reason,
                    "best_validation_loss": float(best.get("validation_loss", np.nan)),
                    "configuration_hash": config_hash,
                    "completion_timestamp": utc_now(),
                }, paths["complete"])
                print(f"Completed stage recovered from {reason}: skipping")
                return "complete", None

    # 3. Exact mid-epoch resume. Ignore an end-of-epoch step checkpoint when
    # history already contains that completed epoch.
    step = load_valid_checkpoint(paths["step"], config_hash)
    if step is not None:
        step_epoch = int(step.get("epoch", 0))
        running = step.get("running_metrics")
        history_has_epoch = (not history.empty and step_epoch in set(history.epoch.astype(int)))
        if running is not None or not history_has_epoch:
            print("Exact step resume")
            return "step", step

    # 4. Exact epoch resume.
    if last is not None:
        print("Exact epoch resume")
        return "epoch", last

    # 5. Legacy best-only checkpoint.
    best = load_valid_checkpoint(paths["best"], config_hash)
    if best is not None:
        print("Legacy best-checkpoint resume")
        return "best", best

    print("Starting new stage")
    return "new", None

def run_resume_smoke_test():
    import tempfile
    with tempfile.TemporaryDirectory() as directory:
        cfg_hash = stable_hash({"smoke": True})
        paths = stage_paths(directory)
        atomic_torch_save({"config_hash": cfg_hash, "epoch": 2}, paths["last"])
        mode, recovered = resume_decision(directory, cfg_hash, 3)
        assert mode == "epoch" and recovered["epoch"] == 2
        history = pd.DataFrame([
            {"epoch": 1, "stage": "joint", "completed_at": "2026-01-01", "value": 2},
            {"epoch": 1, "stage": "joint", "completed_at": "2026-01-02", "value": 1}])
        save_history(history, paths["history"])
        assert len(pd.read_csv(paths["history"])) == 1
    print("Atomic checkpoint and history-resume smoke tests passed.")


## 10 — Optimizers, training loops, and three-stage orchestration


In [ ]:
def amp_context():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)


def set_trainable_stage(model, stage):
    for name, parameter in model.named_parameters():
        parameter.requires_grad = stage != "mixer_warmup" or name.startswith("conditioners.")


def build_optimizer(model, cfg: Config):
    buckets = {"backbone": [], "projection": [], "core": []}
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        key = "core" if ".core." in name else "projection" if name.startswith("conditioners.") else "backbone"
        buckets[key].append(parameter)
    groups = []
    for name, lr in (("backbone", cfg.backbone_lr), ("projection", cfg.projection_lr),
                     ("core", cfg.core_lr)):
        if buckets[name]:
            groups.append({"params": buckets[name], "lr": lr, "name": name})
    optimizer = torch.optim.AdamW(groups, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
    return optimizer, scheduler


def gradient_norm(model, selector):
    total = sum(float(parameter.grad.detach().float().square().sum())
                for name, parameter in model.named_parameters()
                if selector(name) and parameter.grad is not None)
    return math.sqrt(total)


def make_checkpoint(model, optimizer, scheduler, scaler, sampler, early, epoch, global_step,
                    stage, variant, seed, cfg, config_hash, history, running=None, batch_index=None):
    return {
        "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(), "scaler_state": scaler.state_dict(),
        "sampler_state": sampler.state_dict(), "early_stopping": early.state_dict(),
        "epoch": int(epoch), "global_step": int(global_step), "batch_index": batch_index,
        "running_metrics": running, "history": clean_history(history).to_dict("records"),
        "rng_state": rng_state(), "stage": stage, "variant": variant, "seed": int(seed),
        "config": asdict(cfg), "config_hash": config_hash,
        "parameters": count_trainable(model), "quantum_resources": quantum_resource_report(model),
        "saved_at": utc_now(),
    }


def restore_training_checkpoint(checkpoint, model, optimizer, scheduler, scaler, sampler, early):
    model.load_state_dict(checkpoint["model_state"])
    if checkpoint.get("optimizer_state"):
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        scaler.load_state_dict(checkpoint["scaler_state"])
    if checkpoint.get("sampler_state"):
        sampler.load_state_dict(checkpoint["sampler_state"])
    if checkpoint.get("early_stopping"):
        early.load_state_dict(checkpoint["early_stopping"])
    if checkpoint.get("rng_state"):
        restore_rng_state(checkpoint["rng_state"])


def blank_totals():
    return {key: 0.0 for key in ("total", "complex", "magnitude", "phase", "mrstft", "sisdr")}


def train_epoch(model, loader, sampler, criterion, optimizer, scaler, cfg, epoch, global_step,
                step_callback=None, initial_running=None):
    model.train()
    totals = blank_totals() if initial_running is None else dict(initial_running["totals"])
    batches = 0 if initial_running is None else int(initial_running["batches"])
    core_gradient_total = backbone_gradient_total = 0.0
    progress = tqdm(loader, desc=f"Train epoch {epoch}")
    for batch_index, batch in enumerate(progress):
        noisy = batch["noisy"].to(DEVICE, non_blocking=True)
        clean = batch["clean"].to(DEVICE, non_blocking=True)
        snr = batch["snr_db"].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with amp_context():
            enhanced, enhanced_stft = model(noisy)
            clean_stft = model.stft(clean.float())
            losses = criterion(enhanced, enhanced_stft, clean, clean_stft, snr)
        scaler.scale(losses["total"]).backward()
        scaler.unscale_(optimizer)
        core_gradient_total += gradient_norm(model, lambda name: ".core." in name)
        backbone_gradient_total += gradient_norm(model, lambda name: not name.startswith("conditioners."))
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()
        global_step += 1
        sampler.mark_batch_complete()
        batches += 1
        for key in totals:
            totals[key] += float(losses[key].detach())
        progress.set_postfix(loss=f"{totals['total'] / batches:.4f}")
        if step_callback and global_step % cfg.checkpoint_every_steps == 0:
            step_callback(epoch, global_step, batch_index, {"totals": totals, "batches": batches})
    result = {f"train_{key}": value / max(batches, 1) for key, value in totals.items()}
    result["qnn_gradient_norm"] = core_gradient_total / max(batches, 1)
    result["backbone_gradient_norm"] = backbone_gradient_total / max(batches, 1)
    return result, global_step


@torch.no_grad()
def validate_epoch(model, loader, criterion):
    model.eval()
    totals, batches, sisdr_values = blank_totals(), 0, []
    for batch in tqdm(loader, desc="Validation"):
        noisy, clean = batch["noisy"].to(DEVICE), batch["clean"].to(DEVICE)
        snr = batch["snr_db"].to(DEVICE)
        with amp_context():
            enhanced, enhanced_stft = model(noisy)
            clean_stft = model.stft(clean.float())
            losses = criterion(enhanced, enhanced_stft, clean, clean_stft, snr)
        for key in totals:
            totals[key] += float(losses[key])
        sisdr_values.extend(si_sdr_per_sample(enhanced.float(), clean.float()).cpu().tolist())
        batches += 1
    result = {f"validation_{key}": value / max(batches, 1) for key, value in totals.items()}
    result["validation_sisdr"] = float(np.mean(sisdr_values))
    return result


def conditioner_strengths(model):
    return {name: float((0.25 * torch.sigmoid(module.logit_strength)).detach().cpu())
            for name, module in model.conditioners.items()}


def fit_stage(model, train_frame, validation_frame, run_dir, cfg, seed, variant, stage,
              maximum_epochs, patience):
    metadata = {"seed": seed, "variant": variant, "stage": stage}
    config_hash, paths = write_stage_config(run_dir, cfg, metadata)
    mode, checkpoint = resume_decision(run_dir, config_hash, maximum_epochs)
    if mode == "complete":
        best = load_valid_checkpoint(paths["best"], config_hash, DEVICE)
        if best is None:
            raise RuntimeError(f"Completed stage lacks valid best checkpoint: {run_dir}")
        model.load_state_dict(best["model_state"])
        history = clean_history(pd.read_csv(paths["history"])) if paths["history"].exists() else pd.DataFrame()
        return model, history, paths["best"]

    set_trainable_stage(model, stage)
    model = model.to(DEVICE)
    criterion = CompositeEnhancementLoss(cfg)
    optimizer, scheduler = build_optimizer(model, cfg)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    train_loader, sampler = make_train_loader(train_frame, seed, stage, cfg)
    validation_loader = make_validation_loader(validation_frame, seed, cfg)
    early = EarlyStopping(patience, cfg.min_delta)
    history = pd.DataFrame()
    start_epoch, global_step, initial_running = 1, 0, None

    if checkpoint is not None:
        restore_training_checkpoint(checkpoint, model, optimizer, scheduler, scaler, sampler, early)
        history = clean_history(pd.DataFrame(checkpoint.get("history", [])))
        global_step = int(checkpoint.get("global_step", 0))
        if mode == "step":
            start_epoch = int(checkpoint["epoch"])
            initial_running = checkpoint.get("running_metrics")
            if sampler.cursor >= len(sampler.batches):
                start_epoch += 1
                initial_running = None
        else:
            start_epoch = int(checkpoint["epoch"]) + 1
            sampler.set_epoch(start_epoch, 0)

    completion_reason = "max_epochs"
    for epoch in range(start_epoch, maximum_epochs + 1):
        if sampler.epoch != epoch:
            sampler.set_epoch(epoch, 0)
        epoch_started = time.perf_counter()

        def save_step(current_epoch, current_step, batch_index, running):
            payload = make_checkpoint(model, optimizer, scheduler, scaler, sampler, early,
                                      current_epoch, current_step, stage, variant, seed, cfg,
                                      config_hash, history, running, batch_index)
            atomic_torch_save(payload, paths["step"])
            atomic_json_save({"status": "running", "epoch": current_epoch,
                              "global_step": current_step, "sampler": sampler.state_dict(),
                              "updated_at": utc_now()}, paths["state"])

        train_metrics, global_step = train_epoch(
            model, train_loader, sampler, criterion, optimizer, scaler, cfg, epoch, global_step,
            save_step, initial_running=initial_running)
        initial_running = None
        validation_metrics = validate_epoch(model, validation_loader, criterion)
        scheduler.step(validation_metrics["validation_total"])
        improved, should_stop = early.update(validation_metrics["validation_total"], epoch)
        row = {
            "epoch": epoch, "seed": seed, "variant": variant, "stage": stage,
            "candidate_id": stable_hash({"config": asdict(cfg)})[:12],
            **train_metrics, **validation_metrics,
            "validation_pesq": np.nan, "validation_estoi": np.nan,
            "learning_rates": json.dumps({group.get("name", str(index)): group["lr"]
                                           for index, group in enumerate(optimizer.param_groups)}),
            "conditioner_strengths": json.dumps(conditioner_strengths(model)),
            "epoch_seconds": time.perf_counter() - epoch_started, "completed_at": utc_now(),
        }
        history = clean_history(pd.concat([history, pd.DataFrame([row])], ignore_index=True))
        save_history(history, paths["history"])
        payload = make_checkpoint(model, optimizer, scheduler, scaler, sampler, early, epoch,
                                  global_step, stage, variant, seed, cfg, config_hash, history)
        payload.update({"validation_loss": validation_metrics["validation_total"],
                        "validation_sisdr": validation_metrics["validation_sisdr"]})
        atomic_torch_save(payload, paths["last"])
        final_step = copy.deepcopy(payload)
        final_step["running_metrics"] = None
        final_step["sampler_state"]["cursor"] = len(sampler.batches)
        atomic_torch_save(final_step, paths["step"])
        if improved:
            atomic_torch_save(payload, paths["best"])
        print(f"epoch={epoch} train={train_metrics['train_total']:.5f} "
              f"val={validation_metrics['validation_total']:.5f} "
              f"SI-SDR={validation_metrics['validation_sisdr']:.3f}")
        if should_stop:
            completion_reason = "early_stopping"
            break

    atomic_json_save({"last_completed_epoch": int(history.epoch.max()), "best_epoch": early.best_epoch,
                      "completion_reason": completion_reason, "best_validation_loss": early.best,
                      "configuration_hash": config_hash, "completion_timestamp": utc_now()},
                     paths["complete"])
    best = load_valid_checkpoint(paths["best"], config_hash, DEVICE)
    model.load_state_dict(best["model_state"])
    return model, history, paths["best"]


def load_backbone_checkpoint(model, checkpoint_path):
    checkpoint = safe_torch_load(checkpoint_path, map_location="cpu")
    missing, unexpected = model.load_state_dict(checkpoint["model_state"], strict=False)
    if unexpected or any(not name.startswith("conditioners.") for name in missing):
        raise RuntimeError(f"Unexpected backbone mismatch: missing={missing}, unexpected={unexpected}")
    expected_hash = tensor_dict_hash(
        checkpoint["model_state"], exclude=lambda name: name.startswith("conditioners."))
    if backbone_hash(model) != expected_hash:
        raise RuntimeError("Loaded conditioner arm does not match the shared backbone hash")
    return model


def train_shared_baseline(frames, seed, run_root, cfg=CFG, epochs=None):
    seed_everything(seed)
    model = QuantumUNet("baseline", cfg)
    stage_dir = Path(run_root) / f"seed_{seed}" / "shared_pretrain" / "baseline"
    return fit_stage(model, frames["train"], frames["val"], stage_dir, cfg, seed,
                     "baseline", "shared_pretrain", epochs or cfg.shared_pretrain_epochs,
                     cfg.shared_pretrain_patience)


def train_conditioned_arm(frames, seed, variant, cfg, backbone_checkpoint, run_root,
                          warmup_epochs=None, joint_epochs=None):
    seed_everything(seed)
    model = reset_conditioner_shell(QuantumUNet(variant, cfg), seed)
    load_backbone_checkpoint(model, backbone_checkpoint)
    warmup_dir = Path(run_root) / f"seed_{seed}" / "mixer_warmup" / variant
    model, _, _ = fit_stage(model, frames["train"], frames["val"], warmup_dir, cfg, seed, variant,
                            "mixer_warmup", warmup_epochs or cfg.mixer_warmup_epochs,
                            cfg.mixer_warmup_patience)
    joint_dir = Path(run_root) / f"seed_{seed}" / "joint_finetune" / variant
    return fit_stage(model, frames["train"], frames["val"], joint_dir, cfg, seed, variant,
                     "joint_finetune", joint_epochs or cfg.joint_finetune_epochs,
                     cfg.joint_finetune_patience)


## 11 — Signal metrics and resumable per-file evaluation


In [ ]:
def numpy_si_sdr(estimate, reference, eps=1e-8):
    estimate = np.asarray(estimate, dtype=np.float64)
    reference = np.asarray(reference, dtype=np.float64)
    estimate -= estimate.mean()
    reference -= reference.mean()
    scale = np.dot(estimate, reference) / (np.dot(reference, reference) + eps)
    target = scale * reference
    residual = estimate - target
    return float(10 * np.log10((np.sum(target ** 2) + eps) / (np.sum(residual ** 2) + eps)))


def signal_metrics(reference, estimate, sample_rate=CFG.sample_rate):
    from pesq import pesq
    from pystoi import stoi
    length = min(len(reference), len(estimate))
    reference = np.asarray(reference[:length], dtype=np.float32)
    estimate = np.asarray(estimate[:length], dtype=np.float32)
    peak = max(float(np.max(np.abs(reference))), float(np.max(np.abs(estimate))), 1.0)
    reference, estimate = np.clip(reference / peak, -1, 1), np.clip(estimate / peak, -1, 1)
    result = {"si_sdr": numpy_si_sdr(estimate, reference)}
    try:
        result["pesq"] = float(pesq(sample_rate, reference, estimate, "wb"))
    except Exception:
        result["pesq"] = np.nan
    try:
        result["estoi"] = float(stoi(reference, estimate, sample_rate, extended=True))
    except Exception:
        result["estoi"] = np.nan
    return result


def load_model_checkpoint(checkpoint_path, device=DEVICE):
    checkpoint = safe_torch_load(checkpoint_path, map_location="cpu")
    cfg = Config(**checkpoint["config"])
    model = QuantumUNet(checkpoint["variant"], cfg).to(device)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    return model, checkpoint, cfg


def complete_metric_row(row):
    required = {"enhanced_si_sdr", "enhanced_pesq", "enhanced_estoi",
                "delta_si_sdr", "delta_pesq", "delta_estoi"}
    return required.issubset(row.index) and all(pd.notna(row[key]) for key in ("enhanced_si_sdr", "delta_si_sdr"))


def evaluate_checkpoint_resume(checkpoint_path, frame, output_csv, split_name,
                               save_interval=CFG.evaluation_save_interval, quantum_noise=None):
    model, checkpoint, cfg = load_model_checkpoint(checkpoint_path)
    if quantum_noise:
        model.set_quantum_runtime(**quantum_noise)
    variant, seed = checkpoint["variant"], int(checkpoint["seed"])
    output_csv = Path(output_csv)
    if output_csv.exists():
        existing = pd.read_csv(output_csv)
    else:
        existing = pd.DataFrame()
    done = set()
    if not existing.empty:
        valid = existing.apply(complete_metric_row, axis=1)
        done = set(existing.loc[valid, ["variant", "seed", "sample_id"]].astype(str).agg("|".join, axis=1))
    rows = existing.to_dict("records")
    pending = 0
    selected = frame[frame.snr_db.isin(cfg.snr_levels)].sort_values(["snr_db", "sample_id"])
    for sample in tqdm(selected.itertuples(index=False), total=len(selected), desc=f"{split_name}:{variant}:{seed}"):
        key = f"{variant}|{seed}|{sample.sample_id}"
        if key in done:
            continue
        noisy = load_audio_mono(sample.noisy_path, cfg.sample_rate)
        clean = load_audio_mono(sample.clean_path, cfg.sample_rate)
        with torch.inference_mode():
            enhanced = model.enhance(torch.from_numpy(noisy).unsqueeze(0).to(DEVICE)).squeeze(0).cpu().numpy()
        length = min(len(noisy), len(clean), len(enhanced))
        noisy_metrics = signal_metrics(clean[:length], noisy[:length], cfg.sample_rate)
        enhanced_metrics = signal_metrics(clean[:length], enhanced[:length], cfg.sample_rate)
        result = {"variant": variant, "seed": seed, "sample_id": sample.sample_id,
                  "snr_db": sample.snr_db, "split": split_name,
                  "checkpoint": str(checkpoint_path), "evaluated_at": utc_now()}
        for metric in ("si_sdr", "pesq", "estoi"):
            result[f"noisy_{metric}"] = noisy_metrics[metric]
            result[f"enhanced_{metric}"] = enhanced_metrics[metric]
            result[f"delta_{metric}"] = enhanced_metrics[metric] - noisy_metrics[metric]
        if quantum_noise:
            result["quantum_noise"] = json.dumps(quantum_noise, sort_keys=True)
        rows.append(result)
        pending += 1
        if pending % save_interval == 0:
            atomic_csv_save(pd.DataFrame(rows).drop_duplicates(["variant", "seed", "sample_id"], keep="last"),
                            output_csv)
    result = pd.DataFrame(rows).drop_duplicates(["variant", "seed", "sample_id"], keep="last")
    atomic_csv_save(result, output_csv)
    return result


def screening_validation_subset(frame, cfg: Config):
    return (frame[frame.snr_db.isin(cfg.snr_levels)].sort_values(["snr_db", "sample_id"])
            .groupby("snr_db", group_keys=False).head(cfg.screening_validation_per_snr)
            .reset_index(drop=True))


def summarize_metric_frame(frame):
    return {metric: float(frame[metric].mean()) for metric in
            ("delta_si_sdr", "delta_pesq", "delta_estoi",
             "enhanced_si_sdr", "enhanced_pesq", "enhanced_estoi")}


def evaluate_pair_validation(classical_checkpoint, qnn_checkpoint, validation_frame, output_dir, cfg):
    subset = screening_validation_subset(validation_frame, cfg)
    classical = evaluate_checkpoint_resume(classical_checkpoint, subset,
                                             Path(output_dir) / "classical_validation.csv", "validation")
    qnn = evaluate_checkpoint_resume(qnn_checkpoint, subset,
                                      Path(output_dir) / "qnn_validation.csv", "validation")
    classical_summary, qnn_summary = summarize_metric_frame(classical), summarize_metric_frame(qnn)
    row = {}
    for metric in ("delta_si_sdr", "delta_pesq", "delta_estoi"):
        row[f"classical_{metric}"] = classical_summary[metric]
        row[f"qnn_{metric}"] = qnn_summary[metric]
        row[f"qnn_minus_classical_{metric}"] = qnn_summary[metric] - classical_summary[metric]
    return row


## 12 — Validation-only two-stage architecture and learning-rate screening


In [ ]:
def architecture_candidates(base_cfg: Config = CFG):
    candidates = []
    for depth in (2, 4):
        for entanglement in ("ring", "alternating_ring"):
            for layout in ("bottleneck_only", "bottleneck_and_two_skip_gates"):
                cfg = replace(base_cfg, q_depth=depth, entanglement=entanglement,
                              conditioning_layout=layout)
                candidate_id = stable_hash({"depth": depth, "entanglement": entanglement,
                                            "layout": layout})[:12]
                candidates.append((candidate_id, cfg))
    return candidates


def validate_candidate_feasibility(cfg):
    classical, qnn = build_matched_pair(cfg, seed=CFG.screen_seed)
    parameters = count_trainable(qnn)
    if not cfg.min_parameters <= parameters <= cfg.max_parameters:
        raise ValueError(f"Candidate has {parameters:,} parameters outside "
                         f"[{cfg.min_parameters:,}, {cfg.max_parameters:,}]")
    return {"parameters": parameters, "conditioner_parameters": conditioner_parameters(qnn),
            "core_parameters": core_parameters(qnn), "quantum_calls": qnn.quantum_calls_per_segment}


def validation_ranking(results):
    results = results.copy()
    source_columns = ["qnn_minus_classical_delta_si_sdr", "qnn_minus_classical_delta_pesq",
                      "qnn_minus_classical_delta_estoi"]
    z_columns = []
    for column in source_columns:
        standard_deviation = results[column].std(ddof=0)
        z_column = "z_" + column
        results[z_column] = ((results[column] - results[column].mean()) / standard_deviation
                             if standard_deviation > 0 else 0.0)
        z_columns.append(z_column)
    results["selection_score"] = (0.50 * results[z_columns[0]]
                                  + 0.30 * results[z_columns[1]]
                                  + 0.20 * results[z_columns[2]])
    rejection = ((results.parameters > CFG.max_parameters)
                 | ~np.isfinite(results.selection_score)
                 | (results.qnn_gradient_norm <= 0)
                 | (results.qnn_p95_rtf > CFG.latency_rtf_limit))
    results["rejected"] = rejection
    return results.sort_values(["rejected", "selection_score"], ascending=[True, False]).reset_index(drop=True)


def quick_latency(checkpoint_path, audio_seconds=8, warmup=2, repeats=5):
    model, _, cfg = load_model_checkpoint(checkpoint_path)
    waveform = torch.randn(1, cfg.sample_rate * audio_seconds, device=DEVICE) * 0.02
    with torch.inference_mode():
        for _ in range(warmup):
            model.enhance(waveform)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        timings = []
        for _ in range(repeats):
            started = time.perf_counter()
            model.enhance(waveform)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            timings.append(time.perf_counter() - started)
    return float(np.quantile(timings, 0.95) / audio_seconds)


def best_history_gradient(checkpoint_path):
    checkpoint = safe_torch_load(checkpoint_path, map_location="cpu")
    history = pd.DataFrame(checkpoint.get("history", []))
    return float(history.qnn_gradient_norm.replace([np.inf, -np.inf], np.nan).dropna().mean())


def _screening_row_complete(row, required_columns):
    """Return True only when an existing screening row contains all required outputs."""
    for column in required_columns:
        if column not in row.index:
            return False
        value = row[column]
        if pd.isna(value):
            return False
    return True


def _dedupe_screening_rows(frame, subset):
    if frame is None or frame.empty:
        return pd.DataFrame()
    frame = frame.copy()
    for column in subset:
        if column in frame.columns:
            frame[column] = frame[column].astype(str) if column == "candidate_id" else frame[column]
    return frame.drop_duplicates(subset=subset, keep="last").reset_index(drop=True)


def run_screening_stage1(frames, base_cfg: Config = CFG):
    """Resume/reconstruct Stage 1 without repeating completed candidate work.

    Recovery order:
      1. If a complete ranking exists, load it immediately.
      2. Otherwise load screening_results.csv and reuse every complete candidate row.
      3. Only missing/incomplete candidates are trained/evaluated.
      4. Rebuild screening_rankings.csv atomically at the end.

    This means a kernel crash after all candidates were trained but before the final
    ranking file was written is recovered without retraining the completed candidates.
    """
    screen_root = Path(base_cfg.run_root) / "screening" / "stage1"
    screen_root.mkdir(parents=True, exist_ok=True)
    results_path = screen_root / "screening_results.csv"
    ranking_path = screen_root / "screening_rankings.csv"

    expected_candidates = architecture_candidates(base_cfg)
    expected_ids = {candidate_id for candidate_id, _ in expected_candidates}

    # Fast path: a valid complete ranking already exists.
    if ranking_path.exists():
        try:
            ranking = pd.read_csv(ranking_path)
            ranking_ids = set(ranking["candidate_id"].astype(str)) if "candidate_id" in ranking.columns else set()
            if expected_ids.issubset(ranking_ids):
                print(f"Stage 1 already complete: loading {ranking_path}")
                return ranking
            print("Stage-1 ranking exists but is incomplete; reconstructing from saved results/checkpoints.")
        except Exception as error:
            print(f"Could not read Stage-1 ranking ({error}); reconstructing safely.")

    required_columns = {
        "candidate_id", "q_depth", "entanglement", "conditioning_layout",
        "parameters", "conditioner_parameters", "core_parameters", "quantum_calls",
        "qnn_minus_classical_delta_si_sdr", "qnn_minus_classical_delta_pesq",
        "qnn_minus_classical_delta_estoi", "qnn_gradient_norm", "qnn_p95_rtf",
    }

    if results_path.exists():
        try:
            existing = _dedupe_screening_rows(pd.read_csv(results_path), ["candidate_id"])
        except Exception as error:
            print(f"Could not read existing Stage-1 results ({error}); recovering from checkpoints.")
            existing = pd.DataFrame()
    else:
        existing = pd.DataFrame()

    reusable = {}
    if not existing.empty and "candidate_id" in existing.columns:
        for _, row in existing.iterrows():
            candidate_id = str(row["candidate_id"])
            if candidate_id in expected_ids and _screening_row_complete(row, required_columns):
                reusable[candidate_id] = row.to_dict()

    if reusable:
        print(f"Stage 1: reusing {len(reusable)}/{len(expected_ids)} completed candidate rows from {results_path}")

    # Shared pretraining is itself resume-safe. If already complete this returns immediately.
    _, _, baseline_checkpoint = train_shared_baseline(
        frames,
        base_cfg.screen_seed,
        Path(base_cfg.run_root) / "screening" / "shared",
        base_cfg,
        epochs=base_cfg.shared_pretrain_epochs,
    )

    rows_by_id = dict(reusable)

    for candidate_id, cfg in expected_candidates:
        if candidate_id in rows_by_id:
            print(f"Stage-1 candidate {candidate_id}: COMPLETE -> reused")
            continue

        print(
            f"\nStage-1 candidate {candidate_id}: RESUME/RUN "
            f"depth={cfg.q_depth}, entanglement={cfg.entanglement}, "
            f"layout={cfg.conditioning_layout}"
        )

        feasibility = validate_candidate_feasibility(cfg)
        candidate_root = screen_root / f"candidate_{candidate_id}"
        checkpoints = {}

        for variant in ("classical", "qnn"):
            _, _, checkpoints[variant] = train_conditioned_arm(
                frames,
                base_cfg.screen_seed,
                variant,
                cfg,
                baseline_checkpoint,
                candidate_root,
                warmup_epochs=cfg.screen_warmup_epochs,
                joint_epochs=cfg.screen_joint_epochs,
            )

        metrics = evaluate_pair_validation(
            checkpoints["classical"],
            checkpoints["qnn"],
            frames["val"],
            candidate_root / "validation",
            cfg,
        )

        row = {
            "stage": 1,
            "candidate_id": candidate_id,
            "q_depth": cfg.q_depth,
            "entanglement": cfg.entanglement,
            "conditioning_layout": cfg.conditioning_layout,
            **feasibility,
            **metrics,
            "qnn_gradient_norm": best_history_gradient(checkpoints["qnn"]),
            "qnn_p95_rtf": quick_latency(checkpoints["qnn"]),
        }
        rows_by_id[candidate_id] = row

        partial = pd.DataFrame(list(rows_by_id.values()))
        partial = _dedupe_screening_rows(partial, ["candidate_id"])
        atomic_csv_save(partial, results_path)

    results = pd.DataFrame(list(rows_by_id.values()))
    results = _dedupe_screening_rows(results, ["candidate_id"])

    missing_ids = expected_ids - set(results["candidate_id"].astype(str))
    if missing_ids:
        raise RuntimeError(f"Stage 1 recovery finished with missing candidates: {sorted(missing_ids)}")

    atomic_csv_save(results, results_path)
    ranking = validation_ranking(results)
    atomic_csv_save(ranking, ranking_path)
    print(f"Stage 1 complete/recovered: wrote {ranking_path}")
    return ranking


def run_screening_stage2(frames, stage1_ranking, base_cfg: Config = CFG):
    """Resume Stage 2 at candidate+seed granularity and rebuild ranking safely."""
    stage2_root = Path(base_cfg.run_root) / "screening" / "stage2"
    stage2_root.mkdir(parents=True, exist_ok=True)
    results_path = stage2_root / "screening_results.csv"
    ranking_path = stage2_root / "screening_rankings.csv"

    top = stage1_ranking.loc[~stage1_ranking.rejected].head(2).copy()
    expected_keys = {
        (str(architecture.candidate_id), int(seed))
        for architecture in top.itertuples(index=False)
        for seed in base_cfg.screen_confirmation_seeds
    }

    # Fast path for a completed ranking.
    if ranking_path.exists():
        try:
            ranking = pd.read_csv(ranking_path)
            ranking_ids = set(ranking["candidate_id"].astype(str)) if "candidate_id" in ranking.columns else set()
            expected_arch_ids = {key[0] for key in expected_keys}
            if expected_arch_ids.issubset(ranking_ids):
                print(f"Stage 2 already complete: loading {ranking_path}")
                return ranking
            print("Stage-2 ranking exists but is incomplete; resuming from saved results/checkpoints.")
        except Exception as error:
            print(f"Could not read Stage-2 ranking ({error}); resuming safely.")

    required_columns = {
        "candidate_id", "seed", "q_depth", "entanglement", "conditioning_layout",
        "classical_core_lr", "qnn_core_lr", "parameters", "qnn_gradient_norm", "qnn_p95_rtf",
        "classical_delta_si_sdr", "classical_delta_pesq", "classical_delta_estoi",
        "qnn_delta_si_sdr", "qnn_delta_pesq", "qnn_delta_estoi",
        "qnn_minus_classical_delta_si_sdr", "qnn_minus_classical_delta_pesq",
        "qnn_minus_classical_delta_estoi",
    }

    if results_path.exists():
        try:
            existing = _dedupe_screening_rows(pd.read_csv(results_path), ["candidate_id", "seed"])
        except Exception as error:
            print(f"Could not read existing Stage-2 results ({error}); recovering from checkpoints.")
            existing = pd.DataFrame()
    else:
        existing = pd.DataFrame()

    rows_by_key = {}
    if not existing.empty and {"candidate_id", "seed"}.issubset(existing.columns):
        for _, row in existing.iterrows():
            key = (str(row["candidate_id"]), int(row["seed"]))
            if key in expected_keys and _screening_row_complete(row, required_columns):
                rows_by_key[key] = row.to_dict()

    if rows_by_key:
        print(f"Stage 2: reusing {len(rows_by_key)}/{len(expected_keys)} completed candidate-seed rows")

    for architecture in top.itertuples(index=False):
        candidate_id = str(architecture.candidate_id)

        for seed in base_cfg.screen_confirmation_seeds:
            key = (candidate_id, int(seed))
            if key in rows_by_key:
                print(f"Stage-2 candidate {candidate_id}, seed {seed}: COMPLETE -> reused")
                continue

            print(f"\nStage-2 candidate {candidate_id}, seed {seed}: RESUME/RUN")

            _, _, baseline_checkpoint = train_shared_baseline(
                frames,
                seed,
                Path(base_cfg.run_root) / "screening" / "shared",
                base_cfg,
                epochs=base_cfg.shared_pretrain_epochs,
            )

            arm_candidates = {"classical": [], "qnn": []}

            for variant in ("classical", "qnn"):
                for core_lr in (3e-4, 5e-4, 8e-4):
                    cfg = replace(
                        base_cfg,
                        q_depth=int(architecture.q_depth),
                        entanglement=architecture.entanglement,
                        conditioning_layout=architecture.conditioning_layout,
                        core_lr=core_lr,
                    )
                    run_root = (
                        stage2_root
                        / f"candidate_{candidate_id}"
                        / f"seed_{seed}_search"
                        / f"lr_{core_lr:.0e}"
                    )

                    _, _, checkpoint = train_conditioned_arm(
                        frames,
                        seed,
                        variant,
                        cfg,
                        baseline_checkpoint,
                        run_root,
                        warmup_epochs=cfg.screen_warmup_epochs,
                        joint_epochs=cfg.screen_joint_epochs,
                    )

                    output = run_root / variant / "validation.csv"
                    metrics = evaluate_checkpoint_resume(
                        checkpoint,
                        screening_validation_subset(frames["val"], cfg),
                        output,
                        "validation",
                    )
                    summary = summarize_metric_frame(metrics)
                    arm_candidates[variant].append(
                        {
                            "variant": variant,
                            "core_lr": core_lr,
                            "checkpoint": str(checkpoint),
                            **summary,
                        }
                    )

            # Equivalent validation-only LR search budget for both arms.
            selected = {}
            for variant, values in arm_candidates.items():
                selected[variant] = sorted(
                    values,
                    key=lambda row: (
                        row["delta_si_sdr"],
                        row["delta_pesq"],
                        row["delta_estoi"],
                    ),
                    reverse=True,
                )[0]

            row = {
                "stage": 2,
                "candidate_id": candidate_id,
                "seed": seed,
                "q_depth": int(architecture.q_depth),
                "entanglement": architecture.entanglement,
                "conditioning_layout": architecture.conditioning_layout,
                "classical_core_lr": selected["classical"]["core_lr"],
                "qnn_core_lr": selected["qnn"]["core_lr"],
                "parameters": int(architecture.parameters),
                "qnn_gradient_norm": best_history_gradient(selected["qnn"]["checkpoint"]),
                "qnn_p95_rtf": quick_latency(selected["qnn"]["checkpoint"]),
            }

            for metric in ("delta_si_sdr", "delta_pesq", "delta_estoi"):
                row[f"classical_{metric}"] = selected["classical"][metric]
                row[f"qnn_{metric}"] = selected["qnn"][metric]
                row[f"qnn_minus_classical_{metric}"] = (
                    selected["qnn"][metric] - selected["classical"][metric]
                )

            rows_by_key[key] = row
            partial = pd.DataFrame(list(rows_by_key.values()))
            partial = _dedupe_screening_rows(partial, ["candidate_id", "seed"])
            atomic_csv_save(partial, results_path)

    results = pd.DataFrame(list(rows_by_key.values()))
    results = _dedupe_screening_rows(results, ["candidate_id", "seed"])

    present_keys = set(zip(results["candidate_id"].astype(str), results["seed"].astype(int)))
    missing_keys = expected_keys - present_keys
    if missing_keys:
        raise RuntimeError(f"Stage 2 recovery finished with missing candidate/seed rows: {sorted(missing_keys)}")

    atomic_csv_save(results, results_path)

    per_architecture = (
        results.groupby(
            ["candidate_id", "q_depth", "entanglement", "conditioning_layout"],
            as_index=False,
        )
        .agg(
            {
                "parameters": "first",
                "qnn_gradient_norm": "mean",
                "qnn_p95_rtf": "mean",
                "qnn_minus_classical_delta_si_sdr": "mean",
                "qnn_minus_classical_delta_pesq": "mean",
                "qnn_minus_classical_delta_estoi": "mean",
                "classical_core_lr": "median",
                "qnn_core_lr": "median",
            }
        )
    )

    ranking = validation_ranking(per_architecture)
    atomic_csv_save(ranking, ranking_path)
    print(f"Stage 2 complete/recovered: wrote {ranking_path}")
    return ranking


def freeze_configuration(stage2_ranking, base_cfg: Config = CFG, force=False):
    """Freeze the validation-selected winner exactly once.

    Re-running the notebook returns the existing frozen configuration instead of
    changing frozen_at/hash. If locked-test outputs already exist, refreezing to
    a different winner is forbidden.
    """
    acceptable = stage2_ranking.loc[~stage2_ranking.rejected]
    if acceptable.empty:
        raise RuntimeError("No screening candidate passed the preregistered feasibility guards")
    winner = acceptable.iloc[0]
    frozen_path = Path(base_cfg.run_root) / "frozen_configuration.json"
    hash_path = frozen_path.with_name("frozen_configuration_hash.txt")

    if frozen_path.exists() and hash_path.exists() and not force:
        existing_cfg, existing_payload, existing_hash = load_frozen_configuration(base_cfg)
        existing_candidate = str(existing_payload.get("candidate_id"))
        current_candidate = str(winner.candidate_id)
        if existing_candidate != current_candidate:
            test_metrics = Path(base_cfg.run_root) / "test_evaluation" / "per_file_test_metrics.csv"
            if test_metrics.exists():
                raise RuntimeError(
                    "A different Stage-2 winner is now ranked first, but locked-test evaluation "
                    "has already started. Refreezing is forbidden. Preserve the existing frozen configuration."
                )
            raise RuntimeError(
                f"Frozen candidate {existing_candidate} differs from current Stage-2 winner "
                f"{current_candidate}. Delete the frozen files only if you intentionally want to "
                "restart confirmation BEFORE any locked-test evaluation."
            )
        print("Configuration already frozen; reusing existing hash:", existing_hash)
        return existing_cfg, existing_payload, existing_hash

    frozen = replace(base_cfg, q_depth=int(winner.q_depth), entanglement=winner.entanglement,
                     conditioning_layout=winner.conditioning_layout,
                     core_lr=float(winner.qnn_core_lr))
    payload = {"frozen_at": utc_now(), "selection_source": "validation_only",
               "candidate_id": winner.candidate_id, "selection_score": float(winner.selection_score),
               "config": asdict(frozen), "classical_selected_lr": float(winner.classical_core_lr),
               "qnn_selected_lr": float(winner.qnn_core_lr)}
    atomic_json_save(payload, frozen_path)
    frozen_hash = stable_hash(payload)
    atomic_text_save(frozen_hash + "\n", hash_path)
    print("Configuration frozen:", frozen_hash)
    return frozen, payload, frozen_hash

def load_frozen_configuration(base_cfg: Config = CFG):
    path = Path(base_cfg.run_root) / "frozen_configuration.json"
    hash_path = path.with_name("frozen_configuration_hash.txt")
    if not path.exists() or not hash_path.exists():
        raise RuntimeError("No frozen configuration exists; locked-test evaluation is forbidden")
    payload = json.loads(path.read_text())
    expected = hash_path.read_text().strip()
    actual = stable_hash(payload)
    if expected != actual:
        raise RuntimeError("Frozen configuration hash mismatch")
    return Config(**payload["config"]), payload, actual


## 13 — Three-seed confirmation and locked-test gate


In [ ]:
def run_three_seed_confirmation(frames, frozen_cfg, frozen_payload):
    confirmation_root = Path(frozen_cfg.run_root) / "confirmation"
    rows = []
    for seed in frozen_cfg.seeds:
        baseline, baseline_history, baseline_checkpoint = train_shared_baseline(
            frames, seed, confirmation_root, frozen_cfg)
        baseline_validation = evaluate_checkpoint_resume(
            baseline_checkpoint, frames["val"],
            confirmation_root / "validation" / f"seed_{seed}_baseline.csv", "validation")
        baseline_metrics = summarize_metric_frame(baseline_validation)
        rows.append({"seed": seed, "variant": "baseline", "checkpoint": str(baseline_checkpoint),
                     "parameters": count_trainable(baseline),
                     "best_validation_loss": float(baseline_history.validation_total.min()),
                     **{f"validation_{key}": value for key, value in baseline_metrics.items()}})
        for variant in ("classical", "qnn"):
            selected_lr = (float(frozen_payload["classical_selected_lr"]) if variant == "classical"
                           else float(frozen_payload["qnn_selected_lr"]))
            arm_cfg = replace(frozen_cfg, core_lr=selected_lr)
            model, history, checkpoint = train_conditioned_arm(
                frames, seed, variant, arm_cfg, baseline_checkpoint, confirmation_root)
            validation = evaluate_checkpoint_resume(
                checkpoint, frames["val"],
                confirmation_root / "validation" / f"seed_{seed}_{variant}.csv", "validation")
            validation_metrics = summarize_metric_frame(validation)
            rows.append({"seed": seed, "variant": variant, "checkpoint": str(checkpoint),
                         "parameters": count_trainable(model),
                         "best_validation_loss": float(history.validation_total.min()),
                         "core_lr": selected_lr, "final_backbone_hash": backbone_hash(model),
                         **{f"validation_{key}": value for key, value in validation_metrics.items()}})
        seed_rows = [row for row in rows if row["seed"] == seed and row["variant"] != "baseline"]
        assert seed_rows[0]["parameters"] == seed_rows[1]["parameters"]
        atomic_csv_save(pd.DataFrame(rows), confirmation_root / "confirmation_summary.csv")
    return pd.DataFrame(rows)


def verify_confirmation_complete(summary, cfg):
    expected = {(seed, variant) for seed in cfg.seeds
                for variant in ("baseline", "classical", "qnn")}
    actual = set(zip(summary.seed.astype(int), summary.variant))
    if actual != expected:
        raise RuntimeError(f"Confirmation incomplete: missing={expected - actual}")
    for checkpoint_path in summary.checkpoint:
        checkpoint_path = Path(checkpoint_path)
        if not checkpoint_path.exists() or not (checkpoint_path.parent / "complete.json").exists():
            raise RuntimeError(f"Incomplete confirmation stage: {checkpoint_path.parent}")
    paired = summary[summary.variant.isin(["classical", "qnn"])].pivot(
        index="seed", columns="variant", values="parameters")
    if not (paired.classical == paired.qnn).all():
        raise RuntimeError("Confirmation models are not parameter matched")
    return True


def run_locked_test_evaluation(frames, base_cfg: Config = CFG):
    frozen_cfg, frozen_payload, frozen_hash = load_frozen_configuration(base_cfg)
    summary_path = Path(frozen_cfg.run_root) / "confirmation" / "confirmation_summary.csv"
    if not summary_path.exists():
        raise RuntimeError("Confirmation summary is absent; test evaluation is forbidden")
    summary = pd.read_csv(summary_path)
    verify_confirmation_complete(summary, frozen_cfg)
    test_root = Path(frozen_cfg.run_root) / "test_evaluation"
    result_path = test_root / "per_file_test_metrics.csv"
    all_results = []
    for row in summary.itertuples(index=False):
        individual = test_root / "individual" / f"seed_{row.seed}_{row.variant}.csv"
        result = evaluate_checkpoint_resume(row.checkpoint, frames["test"], individual, "locked_test")
        all_results.append(result)
        combined = pd.concat(all_results, ignore_index=True)
        atomic_csv_save(combined.drop_duplicates(["variant", "seed", "sample_id"], keep="last"),
                        result_path)
    atomic_json_save({"frozen_configuration_hash": frozen_hash, "test_started_after_freeze": True,
                      "evaluated_at": utc_now(), "rows": len(pd.concat(all_results))},
                     test_root / "evaluation_provenance.json")
    return pd.concat(all_results, ignore_index=True)


## 14 — Seed-aware paired statistics, Holm correction, and success decision


In [ ]:
def bootstrap_paired_mean(values, repeats=CFG.bootstrap_repeats, seed=2026):
    values = np.asarray(pd.Series(values).dropna(), dtype=np.float64)
    rng = np.random.default_rng(seed)
    samples = values[rng.integers(0, len(values), size=(repeats, len(values)))].mean(axis=1)
    return float(values.mean()), float(np.quantile(samples, 0.025)), float(np.quantile(samples, 0.975))


def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    running = 0.0
    count = len(p_values)
    for rank, index in enumerate(order):
        running = max(running, (count - rank) * p_values[index])
        adjusted[index] = min(1.0, running)
    return adjusted


def paired_effects(differences):
    from scipy.stats import rankdata
    differences = np.asarray(differences, dtype=np.float64)
    nonzero = differences[differences != 0]
    cohen_dz = float(differences.mean() / differences.std(ddof=1)) if differences.std(ddof=1) > 0 else np.nan
    ranks = rankdata(np.abs(nonzero))
    positive, negative = ranks[nonzero > 0].sum(), ranks[nonzero < 0].sum()
    rank_biserial = float((positive - negative) / (positive + negative)) if positive + negative else np.nan
    return cohen_dz, rank_biserial


def paired_qnn_analysis(results, grouping="overall"):
    from scipy.stats import wilcoxon
    metric_names = ("delta_si_sdr", "delta_pesq", "delta_estoi")
    averaged = (results[results.variant.isin(["classical", "qnn"])]
                .groupby(["variant", "sample_id", "snr_db"], as_index=False)[list(metric_names)].mean())
    if grouping == "low_snr":
        averaged = averaged[averaged.snr_db.isin([-5, 0, 5])]
    elif grouping == "high_snr":
        averaged = averaged[averaged.snr_db.isin([10, 15, 20])]
    elif grouping != "overall":
        averaged = averaged[averaged.snr_db == float(grouping)]
    rows = []
    for metric in metric_names:
        paired = averaged.pivot(index="sample_id", columns="variant", values=metric).dropna()
        differences = paired.qnn - paired.classical
        mean, low, high = bootstrap_paired_mean(differences)
        one = wilcoxon(differences, alternative="greater", zero_method="wilcox")
        two = wilcoxon(differences, alternative="two-sided", zero_method="wilcox")
        cohen_dz, rank_biserial = paired_effects(differences)
        rows.append({"group": grouping, "metric": metric, "utterances": len(differences),
                     "mean_difference": mean, "median_difference": float(differences.median()),
                     "ci_low": low, "ci_high": high, "one_sided_p": float(one.pvalue),
                     "two_sided_p": float(two.pvalue), "cohen_dz": cohen_dz,
                     "rank_biserial": rank_biserial})
    frame = pd.DataFrame(rows)
    frame["holm_one_sided_p"] = holm_adjust(frame.one_sided_p)
    return frame


def aggregate_test_tables(results):
    metric_columns = ["enhanced_si_sdr", "enhanced_pesq", "enhanced_estoi",
                      "delta_si_sdr", "delta_pesq", "delta_estoi"]
    per_seed_snr = results.groupby(["variant", "seed", "snr_db"])[metric_columns].agg(["mean", "std"])
    overall = results.groupby(["variant", "seed"])[metric_columns].agg(["mean", "std"])
    seed_means = results.groupby(["variant", "seed"])[metric_columns].mean()
    across_seeds = seed_means.groupby("variant").agg(["mean", "std"])
    return per_seed_snr, overall, across_seeds


def decide_qnn_outcome(overall_analysis, range_analysis):
    indexed = overall_analysis.set_index("metric")
    all_means_positive = (indexed.mean_difference > 0).all()
    primary = indexed.loc["delta_si_sdr"]
    primary_supported = primary.ci_low > 0 and primary.holm_one_sided_p < 0.05
    margins = {"delta_si_sdr": -0.20, "delta_pesq": -0.05, "delta_estoi": -0.01}
    no_range_degradation = all(
        row.mean_difference >= margins[row.metric] for row in range_analysis.itertuples(index=False))
    if all_means_positive and primary_supported and no_range_degradation:
        outcome = "full_qnn_success"
    elif primary_supported and no_range_degradation:
        outcome = "strong_partial_qnn_success"
    elif no_range_degradation and abs(primary.mean_difference) < 0.20:
        outcome = "competitive_result"
    else:
        outcome = "no_demonstrated_qnn_advantage"
    return {"outcome": outcome, "all_metric_means_positive": bool(all_means_positive),
            "primary_ci_positive_and_holm_significant": bool(primary_supported),
            "no_practical_range_degradation": bool(no_range_degradation)}


def all_model_difference_table(results):
    metrics = ("delta_si_sdr", "delta_pesq", "delta_estoi")
    pairs = (("qnn", "classical"), ("qnn", "baseline"), ("classical", "baseline"))
    averaged = (results.groupby(["variant", "sample_id", "snr_db"], as_index=False)
                [list(metrics)].mean())
    rows = []
    groups = {"overall": averaged,
              "low_snr": averaged[averaged.snr_db.isin([-5, 0, 5])],
              "high_snr": averaged[averaged.snr_db.isin([10, 15, 20])]}
    groups.update({f"snr_{snr}": averaged[averaged.snr_db == snr] for snr in CFG.snr_levels})
    for group_name, group in groups.items():
        for left, right in pairs:
            for metric in metrics:
                paired = group[group.variant.isin([left, right])].pivot(
                    index="sample_id", columns="variant", values=metric).dropna()
                differences = paired[left] - paired[right]
                rows.append({"group": group_name, "comparison": f"{left}_minus_{right}",
                             "metric": metric, "utterances": len(differences),
                             "mean_difference": float(differences.mean()),
                             "median_difference": float(differences.median())})
    return pd.DataFrame(rows)


def create_final_test_report(test_results, cfg: Config = CFG):
    output_dir = Path(cfg.run_root) / "test_evaluation"
    per_seed_snr, overall, across_seeds = aggregate_test_tables(test_results)
    per_seed_snr.to_csv(output_dir / "per_seed_per_snr_summary.csv")
    overall.to_csv(output_dir / "per_seed_overall_summary.csv")
    across_seeds.to_csv(output_dir / "three_seed_aggregate.csv")
    analyses = [paired_qnn_analysis(test_results, group)
                for group in ("overall", "low_snr", "high_snr", *cfg.snr_levels)]
    paired = pd.concat(analyses, ignore_index=True)
    atomic_csv_save(paired, output_dir / "paired_statistical_analysis.csv")
    atomic_csv_save(all_model_difference_table(test_results),
                    output_dir / "all_model_pairwise_differences.csv")
    decision = decide_qnn_outcome(
        paired[paired.group == "overall"],
        paired[paired.group.isin(["low_snr", "high_snr"])])
    atomic_json_save(decision, output_dir / "final_experiment_summary.json")
    print(json.dumps(decision, indent=2))
    return paired, decision


## 15 — Training, gradient, conditioner, and per-SNR plots


In [ ]:
import matplotlib.pyplot as plt


def plot_training_history(history_csv):
    history = pd.read_csv(history_csv)
    figure, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(history.epoch, history.train_total, label="train")
    axes[0].plot(history.epoch, history.validation_total, label="validation")
    axes[0].set_title("Composite loss")
    axes[0].legend()
    axes[1].plot(history.epoch, history.qnn_gradient_norm, label="mixer core")
    axes[1].plot(history.epoch, history.backbone_gradient_norm, label="backbone")
    axes[1].set_title("Gradient norms")
    axes[1].legend()
    for name in sorted(set().union(*history.conditioner_strengths.map(lambda value: json.loads(value).keys()))):
        axes[2].plot(history.epoch,
                     history.conditioner_strengths.map(lambda value: json.loads(value).get(name, np.nan)),
                     label=name)
    axes[2].set_title("Conditioner strengths")
    axes[2].legend()
    plt.tight_layout()
    return figure


def plot_metric_by_snr(results, metric="delta_si_sdr"):
    summary = results.groupby(["variant", "snr_db"])[metric].agg(["mean", "sem"]).reset_index()
    plt.figure(figsize=(8, 5))
    for variant, group in summary.groupby("variant"):
        plt.errorbar(group.snr_db, group["mean"], yerr=1.96 * group.sem,
                     marker="o", capsize=3, label=variant)
    plt.xlabel("Input SNR [dB]")
    plt.ylabel(metric.replace("_", " ").upper())
    plt.legend()
    plt.tight_layout()


## 16 — Quantum robustness, latency, memory, and listening samples


In [ ]:
@torch.no_grad()
def benchmark_checkpoint(checkpoint_path, durations=(1, 4, 8), warmup=5, repeats=20):
    model, checkpoint, cfg = load_model_checkpoint(checkpoint_path)
    rows = []
    for seconds in durations:
        waveform = torch.randn(1, cfg.sample_rate * seconds, device=DEVICE) * 0.02
        for _ in range(warmup):
            model.enhance(waveform)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        timings = []
        for _ in range(repeats):
            started = time.perf_counter()
            model.enhance(waveform)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            timings.append(time.perf_counter() - started)
        rows.append({
            "variant": model.variant, "seed": checkpoint["seed"], "device": str(DEVICE),
            "audio_seconds": seconds, "mean_latency_ms": 1000 * np.mean(timings),
            "median_latency_ms": 1000 * np.median(timings),
            "p95_latency_ms": 1000 * np.quantile(timings, 0.95),
            "real_time_factor": np.mean(timings) / seconds,
            "peak_gpu_memory_mb": (torch.cuda.max_memory_allocated() / 2**20
                                   if DEVICE.type == "cuda" else np.nan),
            **quantum_resource_report(model),
        })
    return pd.DataFrame(rows)


def run_latency_suite(confirmation_summary, cfg: Config = CFG):
    """Resume-safe latency/resource benchmarking.

    A completed checkpoint is not benchmarked again on notebook restart.
    """
    output = Path(cfg.run_root) / "resource_analysis"
    output.mkdir(parents=True, exist_ok=True)
    result_path = output / "latency_and_memory.csv"
    existing = pd.read_csv(result_path) if result_path.exists() else pd.DataFrame()
    rows = existing.to_dict("records") if not existing.empty else []

    done = set()
    if not existing.empty and {"checkpoint", "audio_seconds"}.issubset(existing.columns):
        done = set(zip(existing["checkpoint"].astype(str), existing["audio_seconds"].astype(int)))

    for checkpoint in confirmation_summary.checkpoint:
        checkpoint = str(checkpoint)
        # benchmark_checkpoint normally returns 1/4/8 s rows together. Skip if all exist.
        expected = {(checkpoint, seconds) for seconds in (1, 4, 8)}
        if expected and expected.issubset(done):
            print(f"Latency already complete: {checkpoint}")
            continue
        result = benchmark_checkpoint(checkpoint)
        if "checkpoint" not in result.columns:
            result = result.copy()
            result["checkpoint"] = checkpoint
        rows.extend(result.to_dict("records"))
        frame = pd.DataFrame(rows)
        dedup_cols = [column for column in ("checkpoint", "audio_seconds") if column in frame.columns]
        if dedup_cols:
            frame = frame.drop_duplicates(dedup_cols, keep="last")
        atomic_csv_save(frame, result_path)
        done = set(zip(frame["checkpoint"].astype(str), frame["audio_seconds"].astype(int))) \
            if {"checkpoint", "audio_seconds"}.issubset(frame.columns) else set()
    return pd.read_csv(result_path)

def quantum_noise_modes():
    modes = [{"mode": "ideal", "shots": None, "readout_error": 0.0,
              "angle_noise_std": 0.0, "depolarizing": 0.0}]
    modes += [{"mode": f"shots_{shots}", "shots": shots, "readout_error": 0.0,
               "angle_noise_std": 0.0, "depolarizing": 0.0}
              for shots in (128, 512, 1024, 4096)]
    modes += [
        {"mode": "readout_1pct", "shots": 1024, "readout_error": 0.01,
         "angle_noise_std": 0.0, "depolarizing": 0.0},
        {"mode": "readout_3pct", "shots": 1024, "readout_error": 0.03,
         "angle_noise_std": 0.0, "depolarizing": 0.0},
        {"mode": "control_0p01rad", "shots": 1024, "readout_error": 0.01,
         "angle_noise_std": 0.01, "depolarizing": 0.0},
        {"mode": "depolarizing_1pct_approx", "shots": 1024, "readout_error": 0.0,
         "angle_noise_std": 0.0, "depolarizing": 0.01},
    ]
    return modes


def run_quantum_robustness(confirmation_summary, test_frame, cfg: Config = CFG, files_per_snr=25):
    output_root = Path(cfg.run_root) / "quantum_robustness"
    subset = (test_frame.sort_values(["snr_db", "sample_id"]).groupby("snr_db", group_keys=False)
              .head(files_per_snr).reset_index(drop=True))
    all_results = []
    for checkpoint in confirmation_summary.query("variant == 'qnn'").checkpoint:
        seed = int(safe_torch_load(checkpoint, map_location="cpu")["seed"])
        for mode in quantum_noise_modes():
            noise = {key: value for key, value in mode.items() if key != "mode"}
            result = evaluate_checkpoint_resume(
                checkpoint, subset, output_root / mode["mode"] / f"seed_{seed}.csv",
                f"quantum_{mode['mode']}", quantum_noise=noise)
            result["mode"] = mode["mode"]
            all_results.append(result)
    combined = pd.concat(all_results, ignore_index=True)
    atomic_csv_save(combined, output_root / "all_quantum_robustness_metrics.csv")
    return combined


def preselect_listening_samples(test_frame, cfg: Config = CFG):
    output = Path(cfg.run_root) / "listening_samples" / "preselected_samples.csv"
    if output.exists():
        return pd.read_csv(output)
    selected = (test_frame[test_frame.snr_db.isin(cfg.snr_levels)]
                .sort_values(["snr_db", "sample_id"]).groupby("snr_db", group_keys=False).head(1))
    atomic_csv_save(selected, output)
    return selected


def export_listening_samples(confirmation_summary, test_frame, cfg: Config = CFG, seed=17):
    selected = preselect_listening_samples(test_frame, cfg)
    root = Path(cfg.run_root) / "listening_samples" / "preselected"
    models = {}
    for variant in ("baseline", "classical", "qnn"):
        checkpoint = confirmation_summary.query("seed == @seed and variant == @variant").iloc[0].checkpoint
        models[variant] = load_model_checkpoint(checkpoint)[0]
    progress_path = root / "export_progress.csv"
    progress = pd.read_csv(progress_path) if progress_path.exists() else pd.DataFrame()
    done = set(progress.get("sample_id", pd.Series(dtype=str)).astype(str))
    rows = progress.to_dict("records")
    for sample in selected.itertuples(index=False):
        if str(sample.sample_id) in done:
            continue
        sample_dir = root / f"snr_{sample.snr_db}" / str(sample.sample_id)
        sample_dir.mkdir(parents=True, exist_ok=True)
        noisy = load_audio_mono(sample.noisy_path, cfg.sample_rate)
        clean = load_audio_mono(sample.clean_path, cfg.sample_rate)
        sf.write(sample_dir / "clean.wav", clean, cfg.sample_rate)
        sf.write(sample_dir / "noisy.wav", noisy, cfg.sample_rate)
        tensor = torch.from_numpy(noisy).unsqueeze(0).to(DEVICE)
        for variant, model in models.items():
            enhanced = model.enhance(tensor).squeeze(0).cpu().numpy()
            sf.write(sample_dir / f"{variant}_enhanced.wav", enhanced, cfg.sample_rate)
        rows.append({"sample_id": sample.sample_id, "snr_db": sample.snr_db,
                     "seed": seed, "exported_at": utc_now()})
        atomic_csv_save(pd.DataFrame(rows), progress_path)
    return pd.DataFrame(rows)


## 17 — Opt-in reduced-data, held-out-noise, and external-dataset studies

These runners are complete but disabled by default. The held-out-noise study retrains without siren and helicopter; it is separate from the main fixed-split claim. Cross-dataset evaluation requires a manifest with clean_path, noisy_path, snr_db, and optionally sample_id.


In [ ]:
def deterministic_fraction(frame, fraction, seed=2026):
    if fraction >= 1:
        return frame.copy()
    selected = [group.sample(frac=fraction, random_state=seed)
                for _, group in frame.groupby(["snr_db", "noise_category"], dropna=False)]
    return pd.concat(selected).sort_values("sample_id").reset_index(drop=True)


def train_secondary_matched_protocol(frames, frozen_cfg, frozen_payload, output_root, seeds=None):
    rows = []
    for seed in seeds or frozen_cfg.seeds:
        baseline, _, baseline_checkpoint = train_shared_baseline(frames, seed, output_root, frozen_cfg)
        rows.append({"seed": seed, "variant": "baseline", "checkpoint": str(baseline_checkpoint)})
        for variant in ("classical", "qnn"):
            lr = (frozen_payload["classical_selected_lr"] if variant == "classical"
                  else frozen_payload["qnn_selected_lr"])
            arm_cfg = replace(frozen_cfg, core_lr=float(lr))
            _, _, checkpoint = train_conditioned_arm(
                frames, seed, variant, arm_cfg, baseline_checkpoint, output_root)
            rows.append({"seed": seed, "variant": variant, "checkpoint": str(checkpoint)})
        atomic_csv_save(pd.DataFrame(rows), Path(output_root) / "summary.csv")
    return pd.DataFrame(rows)


def run_reduced_data_studies(frames, base_cfg: Config = CFG):
    frozen_cfg, payload, _ = load_frozen_configuration(base_cfg)
    summaries = []
    for fraction in frozen_cfg.reduced_data_fractions:
        fraction_frames = dict(frames)
        fraction_frames["train"] = deterministic_fraction(frames["train"], fraction)
        root = Path(frozen_cfg.run_root) / "secondary" / "reduced_data" / f"fraction_{fraction:.2f}"
        summary = train_secondary_matched_protocol(fraction_frames, frozen_cfg, payload, root)
        summary["training_fraction"] = fraction
        summaries.append(summary)
    result = pd.concat(summaries, ignore_index=True)
    atomic_csv_save(result, Path(frozen_cfg.run_root) / "secondary" / "reduced_data" / "all_runs.csv")
    return result


def run_unseen_noise_study(frames, base_cfg: Config = CFG):
    frozen_cfg, payload, _ = load_frozen_configuration(base_cfg)
    held_out = set(frozen_cfg.held_out_noises)
    study_frames = {
        "train": frames["train"][~frames["train"].noise_category.isin(held_out)].reset_index(drop=True),
        "val": frames["val"][~frames["val"].noise_category.isin(held_out)].reset_index(drop=True),
        "test": frames["test"][frames["test"].noise_category.isin(held_out)].reset_index(drop=True),
    }
    root = Path(frozen_cfg.run_root) / "secondary" / "unseen_noise"
    atomic_json_save({"held_out_noise_categories": sorted(held_out),
                      "counts": {name: len(frame) for name, frame in study_frames.items()}},
                     root / "study_definition.json")
    summary = train_secondary_matched_protocol(study_frames, frozen_cfg, payload, root)
    results = [evaluate_checkpoint_resume(
        row.checkpoint, study_frames["test"], root / f"seed_{row.seed}_{row.variant}.csv",
        "held_out_noise_test") for row in summary.itertuples(index=False)]
    combined = pd.concat(results, ignore_index=True)
    atomic_csv_save(combined, root / "per_file_metrics.csv")
    return combined


def load_external_manifest(path):
    frame = pd.read_csv(path)
    required = {"clean_path", "noisy_path", "snr_db"}
    if not required.issubset(frame.columns):
        raise ValueError(f"External manifest requires {sorted(required)}")
    frame = frame.copy()
    base = Path(path).resolve().parent
    for column in ("clean_path", "noisy_path"):
        frame[column] = frame[column].map(
            lambda value: str((base / value).resolve()) if not Path(str(value)).is_absolute()
            else str(Path(str(value)).resolve()))
    if "sample_id" not in frame:
        frame["sample_id"] = [hashlib.sha1(f"{a}|{b}|{s}".encode()).hexdigest()[:20]
                              for a, b, s in zip(frame.clean_path, frame.noisy_path, frame.snr_db)]
    return frame


def run_external_evaluation(confirmation_summary, base_cfg: Config = CFG):
    frozen_cfg, _, _ = load_frozen_configuration(base_cfg)
    if not frozen_cfg.external_manifest:
        print("Cross-dataset evaluation skipped: CFG.external_manifest is not configured.")
        return pd.DataFrame()
    external = load_external_manifest(frozen_cfg.external_manifest)
    root = Path(frozen_cfg.run_root) / "secondary" / "cross_dataset"
    results = [evaluate_checkpoint_resume(
        row.checkpoint, external, root / f"seed_{row.seed}_{row.variant}.csv", "external_test")
        for row in confirmation_summary.itertuples(index=False)]
    combined = pd.concat(results, ignore_index=True)
    atomic_csv_save(combined, root / "per_file_metrics.csv")
    return combined


## 18 — Execution switches and required run order


In [ ]:
# ============================================================
# ONE-RUN AUTO-RESUME PRIMARY PIPELINE
#
# Run this same cell after any kernel/runtime interruption.
# Stage 1 and Stage 2 are both TRUE intentionally: the screening
# functions now detect/reconstruct completed work and only run
# what is actually missing. Existing checkpoints/results are reused.
# ============================================================

RUN_MANIFEST_AUDIT = False       # Reuse existing locked manifests.
RUN_SMOKE_TESTS = False          # Turn True only for debugging/verification.

RUN_SCREENING_STAGE1 = True      # SAFE: loads/reconstructs completed Stage 1; no blind restart.
RUN_SCREENING_STAGE2 = True      # SAFE: resumes only missing Stage-2 candidate/seed work.
FREEZE_WINNER = True             # Idempotent: reuses existing frozen configuration.
RUN_CONFIRMATION = True          # Seeds 17, 42, 123; baseline/classical/QNN.
RUN_LOCKED_TEST = True           # Blocked automatically until confirmation is complete.
RUN_ROBUSTNESS = True            # Finite-shot/readout/control-noise audit.
RUN_LATENCY = True               # Resume-safe latency/resource benchmarking.
RUN_LISTENING_EXPORT = True      # Resume-safe fixed listening samples.

# Large secondary studies remain separate from the main experiment.
RUN_REDUCED_DATA = False
RUN_UNSEEN_NOISE = False
RUN_EXTERNAL_DATASET = False

print("=" * 78)
print("QUNET ONE-RUN AUTO-RESUME PIPELINE")
print("Run directory:", RUN_ROOT)
print("Stage-1 results:", RUN_ROOT / "screening/stage1")
print("Stage-2 results:", RUN_ROOT / "screening/stage2")
print("=" * 78)

# -------------------- DATA --------------------
if RUN_MANIFEST_AUDIT:
    SPLIT_FRAMES, SPLIT_AUDIT = prepare_locked_manifests()
else:
    SPLIT_FRAMES = {
        split: load_split_metadata(split)
        for split in ("train", "val", "test")
    }
    SPLIT_AUDIT = audit_splits(SPLIT_FRAMES)

if RUN_SMOKE_TESTS:
    run_quantum_unit_tests()
    PARAMETER_REPORT = run_model_fairness_smoke_tests()
    run_resume_smoke_test()

# -------------------- STAGE 1 --------------------
# IMPORTANT: A missing screening_rankings.csv no longer means Stage 1 must
# restart. run_screening_stage1() first reuses screening_results.csv rows,
# then resumes only genuinely missing/incomplete candidates, and finally
# reconstructs screening_rankings.csv.
if RUN_SCREENING_STAGE1:
    STAGE1_RANKING = run_screening_stage1(SPLIT_FRAMES, CFG)
else:
    stage1_path = RUN_ROOT / "screening/stage1/screening_rankings.csv"
    if not stage1_path.exists():
        raise FileNotFoundError(
            f"Stage-1 ranking missing: {stage1_path}. "
            "Set RUN_SCREENING_STAGE1=True to reconstruct/resume it safely."
        )
    STAGE1_RANKING = pd.read_csv(stage1_path)

print("\nStage-1 ranking ready:")
display(STAGE1_RANKING.head()) if "display" in globals() else print(STAGE1_RANKING.head())

# -------------------- STAGE 2 --------------------
if RUN_SCREENING_STAGE2:
    STAGE2_RANKING = run_screening_stage2(SPLIT_FRAMES, STAGE1_RANKING, CFG)
else:
    stage2_path = RUN_ROOT / "screening/stage2/screening_rankings.csv"
    if not stage2_path.exists():
        raise FileNotFoundError(
            f"Stage-2 ranking missing: {stage2_path}. "
            "Set RUN_SCREENING_STAGE2=True to resume it safely."
        )
    STAGE2_RANKING = pd.read_csv(stage2_path)

print("\nStage-2 ranking ready:")
display(STAGE2_RANKING.head()) if "display" in globals() else print(STAGE2_RANKING.head())

# -------------------- FREEZE VALIDATION WINNER --------------------
if FREEZE_WINNER:
    FROZEN_CFG, FROZEN_PAYLOAD, FROZEN_HASH = freeze_configuration(
        STAGE2_RANKING, CFG
    )
else:
    FROZEN_CFG, FROZEN_PAYLOAD, FROZEN_HASH = load_frozen_configuration(CFG)

# -------------------- THREE-SEED CONFIRMATION --------------------
if RUN_CONFIRMATION:
    # Reload persisted frozen payload so confirmation always uses the exact
    # validation-selected, hashed configuration.
    FROZEN_CFG, FROZEN_PAYLOAD, FROZEN_HASH = load_frozen_configuration(CFG)
    CONFIRMATION_SUMMARY = run_three_seed_confirmation(
        SPLIT_FRAMES,
        FROZEN_CFG,
        FROZEN_PAYLOAD,
    )
else:
    confirmation_path = RUN_ROOT / "confirmation/confirmation_summary.csv"
    if not confirmation_path.exists():
        raise FileNotFoundError(f"Confirmation summary is missing: {confirmation_path}")
    CONFIRMATION_SUMMARY = pd.read_csv(confirmation_path)

# -------------------- VERIFY BEFORE TEST ACCESS --------------------
if any(
    (
        RUN_LOCKED_TEST,
        RUN_ROBUSTNESS,
        RUN_LATENCY,
        RUN_LISTENING_EXPORT,
        RUN_REDUCED_DATA,
        RUN_UNSEEN_NOISE,
        RUN_EXTERNAL_DATASET,
    )
):
    FROZEN_CFG, FROZEN_PAYLOAD, FROZEN_HASH = load_frozen_configuration(CFG)
    confirmation_path = RUN_ROOT / "confirmation/confirmation_summary.csv"
    if not confirmation_path.exists():
        raise RuntimeError(
            "Confirmation has not completed; downstream evaluation is blocked."
        )
    CONFIRMATION_SUMMARY = pd.read_csv(confirmation_path)
    verify_confirmation_complete(CONFIRMATION_SUMMARY, FROZEN_CFG)

# -------------------- LOCKED TEST --------------------
if RUN_LOCKED_TEST:
    TEST_RESULTS = run_locked_test_evaluation(SPLIT_FRAMES)
    PAIRED_RESULTS, FINAL_DECISION = create_final_test_report(
        TEST_RESULTS,
        FROZEN_CFG,
    )

# -------------------- QNN ROBUSTNESS --------------------
if RUN_ROBUSTNESS:
    QUANTUM_ROBUSTNESS = run_quantum_robustness(
        CONFIRMATION_SUMMARY,
        SPLIT_FRAMES["test"],
        FROZEN_CFG,
    )

# -------------------- LATENCY / RESOURCES --------------------
if RUN_LATENCY:
    LATENCY_RESULTS = run_latency_suite(
        CONFIRMATION_SUMMARY,
        FROZEN_CFG,
    )

# -------------------- LISTENING SAMPLES --------------------
if RUN_LISTENING_EXPORT:
    LISTENING_EXPORTS = export_listening_samples(
        CONFIRMATION_SUMMARY,
        SPLIT_FRAMES["test"],
        FROZEN_CFG,
    )

# -------------------- OPTIONAL SECONDARY STUDIES --------------------
if RUN_REDUCED_DATA:
    REDUCED_DATA_RESULTS = run_reduced_data_studies(SPLIT_FRAMES)

if RUN_UNSEEN_NOISE:
    UNSEEN_NOISE_RESULTS = run_unseen_noise_study(SPLIT_FRAMES)

if RUN_EXTERNAL_DATASET:
    EXTERNAL_RESULTS = run_external_evaluation(CONFIRMATION_SUMMARY)

print("\n" + "=" * 78)
print("PRIMARY QUNET PIPELINE FINISHED SUCCESSFULLY")
print(f"Frozen configuration hash: {FROZEN_HASH}")
print(f"Run directory: {RUN_ROOT}")
print("After any interruption: rerun all definition cells, then rerun THIS cell.")
print("Completed work is loaded/reconstructed/resumed instead of restarted.")
print("=" * 78)


## 19 — Interpretation and reporting rule

Use final_experiment_summary.json without changing the frozen architecture after test evaluation.

- Full QNN success requires all three mean improvements, a positive paired SI-SDR confidence interval with Holm-adjusted significance, and no predefined SNR-range degradation.
- Strong partial success requires supported SI-SDR improvement with the remaining metrics matched, or a predefined low-SNR advantage.
- A competitive result is a parameter-matched tie with a separately predefined robustness or reduced-data benefit.
- Otherwise report no demonstrated QNN advantage.

Simulator timings are exact classical state-vector simulation timings, not physical quantum speed or quantum-hardware evidence. Post-hoc listening examples never replace the preselected samples or complete test tables.


In [ ]:
from pathlib import Path
import shutil
import zipfile
import os

# ============================================================
# SETTINGS
# ============================================================

RUN_DIR = Path(
    "./runs/qunet_multivqc"
)

OUTPUT_DIR = Path(
    "./qunet_results_for_chatgpt"
)

ZIP_PATH = Path(
    "./qunet_results_for_chatgpt.zip"
)

# Set True only if you also want huge model checkpoints in the ZIP.
# For result analysis, keep this False.
INCLUDE_CHECKPOINTS = False

# File types useful for final analysis
KEEP_EXTENSIONS = {
    ".csv",
    ".json",
    ".txt",
    ".log",
    ".png",
    ".jpg",
    ".jpeg",
    ".pdf",
    ".wav",
}

if INCLUDE_CHECKPOINTS:
    KEEP_EXTENSIONS.update({".pt", ".pth"})


# ============================================================
# CHECK
# ============================================================

if not RUN_DIR.exists():
    raise FileNotFoundError(f"Run directory not found:\n{RUN_DIR}")

print("Source run directory:")
print(RUN_DIR)


# ============================================================
# CLEAN OLD EXPORT
# ============================================================

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()


# ============================================================
# COPY RESULT FILES WHILE PRESERVING DIRECTORY STRUCTURE
# ============================================================

copied_files = []

for source_file in RUN_DIR.rglob("*"):

    if not source_file.is_file():
        continue

    # Skip temporary files
    if ".tmp" in source_file.name:
        continue

    suffix = source_file.suffix.lower()

    if suffix not in KEEP_EXTENSIONS:
        continue

    # Preserve folder structure relative to RUN_DIR
    relative_path = source_file.relative_to(RUN_DIR)
    destination = OUTPUT_DIR / relative_path

    destination.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(source_file, destination)
    copied_files.append(destination)


# ============================================================
# CREATE MANIFEST
# ============================================================

manifest_path = OUTPUT_DIR / "EXPORT_MANIFEST.txt"

with open(manifest_path, "w") as f:
    f.write("QUNet MultiVQC Results Export\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Original run directory:\n{RUN_DIR}\n\n")
    f.write(f"Number of copied files: {len(copied_files)}\n\n")
    f.write("Files included:\n")
    f.write("-" * 70 + "\n")

    for file in sorted(copied_files):
        relative = file.relative_to(OUTPUT_DIR)
        size_mb = file.stat().st_size / (1024 ** 2)
        f.write(f"{relative}    [{size_mb:.3f} MB]\n")


# ============================================================
# CREATE ZIP
# ============================================================

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zip_file:

    for file in OUTPUT_DIR.rglob("*"):

        if file.is_file():
            archive_name = file.relative_to(OUTPUT_DIR)
            zip_file.write(file, arcname=archive_name)


# ============================================================
# SUMMARY
# ============================================================

zip_size_mb = ZIP_PATH.stat().st_size / (1024 ** 2)

print("\n" + "=" * 70)
print("EXPORT COMPLETE")
print("=" * 70)

print(f"\nFiles collected : {len(copied_files)}")

print("\nResults folder:")
print(OUTPUT_DIR)

print("\nZIP file:")
print(ZIP_PATH)

print(f"\nZIP size: {zip_size_mb:.2f} MB")

print("\nImportant result files found:")

important_names = [
    "three_seed_aggregate.csv",
    "per_seed_overall_summary.csv",
    "per_seed_per_snr_summary.csv",
    "paired_statistical_analysis.csv",
    "all_model_pairwise_differences.csv",
    "screening_rankings.csv",
]

all_exported = list(OUTPUT_DIR.rglob("*"))

for name in important_names:
    matches = [
        p for p in all_exported
        if p.is_file() and p.name == name
    ]

    if matches:
        for match in matches:
            print(" ✅", match.relative_to(OUTPUT_DIR))
    else:
        print(" ⚠️ ", name, "not found")

print("\nDone.")
print("Upload this file to ChatGPT:")
print(ZIP_PATH)